# GDELT + 네이버 뉴스 모니터링 수집기 v2
## 해상 공급망 교란 — 모니터링 수집 + LLM 분류 + 데일리 리포트

**파이프라인** (12셀):
Cell 1(설정) → Cell 2(GDELT 수집) → Cell 3(네이버 수집) → Cell 4(통계)
→ Cell 5(GDELT dedup) → Cell 6(GDELT LLM 분류)
→ Cell 7(네이버 dedup) → Cell 8(네이버 LLM 분류)
→ Cell 9(저장) → Cell 10(기계적 리포트) → Cell 11(LLM 리포트)

**수집 전략**:
- GDELT: 영문(`en`) 키워드, `English` 기사만
- 네이버: 한국어(`ko`) 키워드, `Korean` 기사만 (환경변수 `NAVER_CLIENT_ID`/`NAVER_CLIENT_SECRET` 필요)

**LLM 분류**: Anthropic API (claude-haiku-4-5) 2단계 — Relevance + KMI 10개 카테고리

**출력**: `gdelt_mon_classified_daily_*.csv` + `naver_mon_classified_daily_*.csv` + `daily_report_*.md/.docx/.xlsx`


In [10]:
pip install python-docx openpyxl

Note: you may need to restart the kernel to use updated packages.


In [13]:
# ============================================================
# Cell 1: 공통 설정 — 날짜 입력 + 수집/저장 헬퍼 함수
# ============================================================

import json, os, hashlib, time, re
import pandas as pd
from datetime import datetime, timedelta, date
from dateutil.parser import parse as dateparse
from gdeltdoc import GdeltDoc, Filters

gd = GdeltDoc()

# ── 날짜 파싱 헬퍼 ──
def _parse(s):
    s = s.strip().lower()
    if s == "today":     return datetime.now().date()
    if s == "yesterday": return (datetime.now() - timedelta(days=1)).date()
    return dateparse(s).date()

# ── 수집 파라미터 ──
MAX_RECORDS = 250
SLEEP_SEC   = 0.5
LANGUAGES   = 'English'             # 영어 전용 (한국어는 네이버에서 수집)

# ── 폴더 구조 ──
MONITOR_DIR = 'monitoring'
CUMUL_DIR   = os.path.join(MONITOR_DIR, 'cumulative')
os.makedirs(CUMUL_DIR, exist_ok=True)

# ── 누적 파일 (cumulative/) ──
CLASSIFIED_CSV       = os.path.join(CUMUL_DIR, 'gdelt_mon_classified.csv')
NAVER_CLASSIFIED_CSV = os.path.join(CUMUL_DIR, 'naver_mon_classified.csv')

# ── 쿼리 파일 ──
MON_QUERY_FILE = 'news_queries_monitoring_v2.json'

# ══════════════════════════════════════════════════════════════
# 수집 기간 입력
# ══════════════════════════════════════════════════════════════
default_end = (datetime.now() - timedelta(days=1)).date()

print("=== GDELT 뉴스 수집기 ===")
print(f"수집 언어: {LANGUAGES}")
print(f"누적 파일: {CLASSIFIED_CSV}")
print("날짜 형식: YYYY-MM-DD  /  today  /  yesterday")
print("Enter 치면 기본값이 적용됩니다.\n")

raw_end = input(f"📅 종료일 [{default_end}]: ").strip()
d_end   = _parse(raw_end) if raw_end else default_end

# ── 날짜별 폴더는 save_daily_csv() 내부에서 각 날짜에 맞게 자동 생성됨 ──

raw_start = input(f"📅 시작일 [{d_end}]: ").strip()
d_start   = _parse(raw_start) if raw_start else d_end

assert d_start <= d_end, f"❌ 시작일({d_start})이 종료일({d_end})보다 뒤입니다!"

collect_dates = []
d = d_start
while d <= d_end:
    collect_dates.append(d)
    d += timedelta(days=1)

print(f"\n수집 기간: {d_start} ~ {d_end} ({len(collect_dates)}일)")

# ── 이미 분류 완료된 날짜 검사 (그룹별 분리) ──
def _get_classified_dates(csv_path):
    """분류 완료 CSV에서 수집일 목록 추출"""
    if not os.path.exists(csv_path):
        return set()
    df = pd.read_csv(csv_path, usecols=['collect_date'], encoding='utf-8-sig')
    dates = set(df['collect_date'].unique())
    del df
    return dates

already = _get_classified_dates(CLASSIFIED_CSV)

dates_to_collect = []
dates_overwrite  = []
OVERWRITE = False

overlap = [d for d in collect_dates if str(d) in already]
if overlap:
    print(f"\n⚠ 이미 분류된 날짜 {len(overlap)}일: {', '.join(str(d) for d in overlap)}")
    ow = input("🔄 덮어쓰기? (y/n) [n]: ").strip().lower()
    OVERWRITE = ow in ('y', 'yes', 'ㅛ')

for d in collect_dates:
    ds = str(d)
    if ds in already:
        if OVERWRITE:
            dates_overwrite.append(d)
            dates_to_collect.append(d)
        else:
            print(f"  건너뜀: {d} (이미 분류됨)")
    else:
        dates_to_collect.append(d)

if not dates_to_collect:
    print(f"\n❌ 수집할 날짜가 없습니다.")
else:
    for d in dates_to_collect:
        tag = " (overwrite)" if d in dates_overwrite else ""
        print(f"  → {d}{tag}")

# ══════════════════════════════════════════════════════════════
# 수집/저장 공통 함수
# ══════════════════════════════════════════════════════════════

def _make_hash(url):
    """URL → MD5 해시 (GDELT/네이버 공통 중복 제거용)"""
    return hashlib.md5(url.encode()).hexdigest()


def load_keywords(query_file, lang='en'):
    """쿼리 JSON에서 키워드 추출 + query_group 매핑.
    lang='en': GDELT 영문 수집용
    lang='ko': 네이버 한국어 수집용
    """
    with open(query_file, 'r', encoding='utf-8') as f:
        config = json.load(f)
    keywords = []
    kw_source = {}
    for qname, qs in config['query_sets'].items():
        for kw in qs.get(lang, []):
            if kw not in kw_source:
                keywords.append(kw)
                kw_source[kw] = qname
    return keywords, kw_source


MIN_KEYWORD_LEN = 5   # GDELT DOC API는 5글자 미만 키워드 거부
MAX_RETRIES     = 2   # ConnectionResetError 등 네트워크 에러 재시도 횟수


def collect_gdelt(keywords, kw_source, target_dates):
    """GDELT 수집 — 키워드당 1회 호출 (language=OR 조건)"""
    daily_results = {}
    daily_stats   = {}
    daily_errors  = {}

    # ── 짧은 키워드 사전 필터링 ──
    skipped_short = [kw for kw in keywords if len(kw) < MIN_KEYWORD_LEN]
    if skipped_short:
        print(f"  ⚠ GDELT 최소 길이({MIN_KEYWORD_LEN}자) 미달 → 스킵: {skipped_short}")
        print(f"    → news_queries_monitoring_v2.json에서 키워드를 길게 수정하세요.")
    valid_keywords = [kw for kw in keywords if len(kw) >= MIN_KEYWORD_LEN]
    n_kw = len(valid_keywords)

    for day_idx, target_date in enumerate(target_dates):
        s_date = target_date.strftime('%Y-%m-%d')
        e_date = (target_date + timedelta(days=1)).strftime('%Y-%m-%d')

        articles  = []
        seen_urls = set()
        errors    = []
        stats     = []
        t0 = time.time()

        print(f"  ── [{day_idx+1}/{len(target_dates)}] {target_date} ({n_kw}키워드) ──")

        for idx, keyword in enumerate(valid_keywords):
            qgroup = kw_source[keyword]
            n_raw = 0
            n_new = 0
            err = None

            for attempt in range(MAX_RETRIES + 1):
                try:
                    f = Filters(
                        keyword=keyword,
                        start_date=s_date,
                        end_date=e_date,
                        num_records=MAX_RECORDS,
                        language=LANGUAGES,
                    )
                    results = gd.article_search(f)

                    if results is not None and len(results) > 0:
                        n_raw = len(results)
                        for _, row in results.iterrows():
                            url = row.get('url', '')
                            url_hash = _make_hash(url)
                            if url_hash not in seen_urls:
                                seen_urls.add(url_hash)
                                articles.append({
                                    'url': url,
                                    'url_hash': url_hash,
                                    'title': row.get('title', ''),
                                    'seendate': row.get('seendate', ''),
                                    'domain': row.get('domain', ''),
                                    'language': row.get('language', ''),
                                    'sourcecountry': row.get('sourcecountry', ''),
                                    'socialimage': row.get('socialimage', ''),
                                    'query_keyword': keyword,
                                    'query_group': qgroup,
                                    'collect_date': str(target_date),
                                })
                                n_new += 1

                    time.sleep(SLEEP_SEC)
                    break  # 성공 → retry 루프 탈출

                except Exception as e:
                    err = str(e)
                    is_network = any(x in err for x in ['ConnectionReset', 'ConnectionAborted',
                                                         'RemoteDisconnected', 'timeout', 'Timeout'])
                    if is_network and attempt < MAX_RETRIES:
                        wait = 3 * (attempt + 1)
                        print(f"      🔄 {keyword}: 네트워크 에러, {wait}초 후 재시도 ({attempt+1}/{MAX_RETRIES})")
                        time.sleep(wait)
                        err = None  # 재시도 성공하면 에러 없앰
                        continue

                    errors.append({'keyword': keyword, 'error': err})
                    if 'Rate' in err or 'limit' in err.lower():
                        print(f"      ⏳ rate limit ({keyword})")
                        time.sleep(5)
                    elif 'too short' in err.lower():
                        print(f"      ⚠ {keyword}: GDELT 키워드 너무 짧음 — JSON에서 수정 필요")
                    else:
                        print(f"      ⚠ {keyword}: {err}")
                    break  # 재시도 불가능한 에러 → 다음 키워드로

            stats.append({
                'keyword': keyword, 'query_group': qgroup,
                'raw': n_raw, 'new': n_new, 'error': err or ''
            })

            if (idx + 1) % 10 == 0:
                elapsed = time.time() - t0
                print(f"      [{idx+1}/{n_kw}] {len(articles)}건  ({elapsed:.0f}초)")

        elapsed = time.time() - t0
        daily_results[target_date] = articles
        daily_stats[target_date]   = stats
        daily_errors[target_date]  = errors

        # skipped 키워드 stats도 추가 (에러 기록용)
        for kw in skipped_short:
            stats.append({
                'keyword': kw, 'query_group': kw_source[kw],
                'raw': 0, 'new': 0, 'error': f'SKIPPED: len={len(kw)} < {MIN_KEYWORD_LEN}'
            })

        n_err = len([e for e in errors if e])
        print(f"      [{n_kw}/{n_kw}] 완료: {len(articles)}건, 에러 {n_err}건 ({elapsed:.0f}초)")

    return daily_results, daily_stats, daily_errors


def save_daily_csv(daily_results, base_prefix):
    """daily CSV를 날짜별 폴더(monitoring/YYYYMMDD/)에 저장.
    base_prefix: 경로 없이 'gdelt_mon' 또는 'naver_mon'만 전달.
    """
    df_all = pd.DataFrame()

    for target_date, articles in daily_results.items():
        date_tag = target_date.strftime('%Y%m%d')
        # ── 날짜별 폴더 자동 생성 ──
        daily_dir = os.path.join(MONITOR_DIR, date_tag)
        os.makedirs(daily_dir, exist_ok=True)
        daily_csv = os.path.join(daily_dir, f'{base_prefix}_daily_{date_tag}.csv')

        if len(articles) == 0:
            print(f"    {target_date}: 0건 — CSV 미생성")
            continue

        df = pd.DataFrame(articles)
        meta_cols = ['url_hash', 'query_keyword', 'query_group', 'collect_date']
        gdelt_cols = [c for c in df.columns if c not in meta_cols]
        df = df[meta_cols + gdelt_cols]

        if 'seendate' in df.columns:
            df = df.sort_values('seendate', ascending=False).reset_index(drop=True)

        df.to_csv(daily_csv, index=False, encoding='utf-8-sig')
        print(f"    {target_date}: {len(df)}건 → {daily_csv}")
        df_all = pd.concat([df_all, df], ignore_index=True)

    return df_all


# ══════════════════════════════════════════════════════════════
# 공통 유틸리티 함수 — dedup / LLM 분류 (모든 셀에서 사용 가능)
# ══════════════════════════════════════════════════════════════

def dedup_by_title(df, group_label):
    """제목 정규화 후 중복 제거."""
    df = df.copy()
    before = len(df)
    df['_title_norm'] = df['title'].str.lower().str.strip()
    df = df.drop_duplicates(subset='_title_norm', keep='first')
    df = df.drop(columns=['_title_norm'])
    after = len(df)
    removed = before - after
    if removed > 0:
        print(f"  제목 중복 제거: {before} → {after}건 (-{removed}건, {removed/before*100:.0f}%)")
    else:
        print(f"  제목 중복 없음: {before}건")
    return df.reset_index(drop=True)

# news_kg_mapping_v3 동일 방식:
# ============================================================

import anthropic

client = anthropic.Anthropic()  # ANTHROPIC_API_KEY 환경변수 자동 사용
MODEL = "claude-haiku-4-5-20251001"

BATCH_SIZE = 20

# ── KMI 모니터링 10개 카테고리 ──
CATEGORIES = {
    "1_Security":       "Security — military threats, geopolitical tensions, chokepoint control, naval operations",
    "2_Safety":         "Safety — vessel accidents, maritime safety incidents, IMO regulations",
    "3_Freight":        "Freight Rates / Transit Volume — freight index changes (BDI, SCFI, etc.), traffic volume shifts",
    "4_PortCargo":      "Port Cargo Volume — import cargo volume, port throughput, port conditions",
    "5_EconFinance":    "Economy / Finance — stock prices, exchange rates, financial index movements, economic indicators",
    "6_Seafood":        "Seafood Trade — export/import value and unit prices of seafood, fishing industry impacts",
    "7_Shipping":       "Shipping Industry — domestic/overseas shipping companies, carrier trends, alternative routes, fleet changes",
    "8_Logistics":      "Logistics Companies — impacts on logistics/forwarding companies, response measures",
    "9_PortCongestion": "Port Congestion — port delays, berth waiting, container dwell time",
    "10_OtherIndustry": "Other Industries — petroleum/naphtha, LNG, fertilizers, bunkering price changes, petrochemical impacts",
}

CATEGORY_KEYS = list(CATEGORIES.keys())


def call_llm_json(prompt, system="Return ONLY valid JSON.", max_tokens=2048, retries=3):
    """Anthropic API 호출 → JSON 파싱 (재시도 포함)"""
    text = ''
    for attempt in range(retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=max_tokens,
                system=system,
                messages=[{"role": "user", "content": prompt}]
            )
            text = resp.content[0].text.strip()
            if text.startswith("```"):
                text = re.sub(r'^```\w*\n?', '', text)
                text = re.sub(r'\n?```$', '', text)
            try:
                return json.loads(text)
            except json.JSONDecodeError as je:
                if 'Extra data' in str(je):
                    decoder = json.JSONDecoder()
                    obj, _ = decoder.raw_decode(text)
                    return obj
                raise
        except (json.JSONDecodeError, Exception) as e:
            if attempt < retries - 1:
                time.sleep(1 * (attempt + 1))
                continue
            print(f"  ⚠ LLM JSON 파싱 실패 ({attempt+1}회): {e}")
            print(f"  Raw: {text[:200]}...")
            return None


# ── 분류 프롬프트 ──

SYSTEM_MSG = (
    "You are a Korean supply chain risk analyst. "
    "Classify news relevance to KOREAN maritime supply chain disruption. "
    "Return ONLY valid JSON array."
)

# ── 카테고리 블록: KG용 (간결) ──

# ── 카테고리 블록: 모니터링용 (상세 분류 기준 — news_monitoring_criteria_en.docx 기반) ──
CAT_BLOCK_MON = """  - 1_Security: International relations changes — transit fee disputes, military deployments, UN resolutions, 
    joint condemnation statements, naval operations, chokepoint control, Iran tensions, Houthi attacks
  - 2_Safety: Vessel accidents, casualties, IMO emergency meetings, maritime safety regulations, 
    Iran communications to IMO, safety incidents at sea
  - 3_Freight: Freight index changes (BDI, SCFI, CCFI, WCI), traffic volume shifts through Suez/Hormuz/Cape/Aden, 
    tanker rate spikes, container freight surcharges, shipping rate trends
  - 4_PortCargo: Import cargo volume changes (crude oil, LNG, petroleum products, refined products), 
    port throughput data, import port status, berth utilization
  - 5_EconFinance: Stock prices (KOSPI, MSCI), VIX, exchange rates (KRW/USD), Brent/WTI oil price movements, 
    financial market reactions in US/Iran/Korea, economic indicators
  - 6_Seafood: Seafood export/import values and unit prices, fishing industry difficulties from oil price rises, 
    coastal/deep-sea fishing company impacts, seafood market inflation
  - 7_Shipping: Domestic shipping company (HMM, Pan Ocean) trends, top 10 global carrier updates (Maersk, MSC, CMA CGM), 
    alternative route strategies, emergency bunker surcharges (EBS), fleet repositioning
  - 8_Logistics: Logistics company impacts (CJ Logistics, Hyundai Glovis), logistics cost changes, 
    operational difficulties, shipper service disruptions, forwarding company responses
  - 9_PortCongestion: Port congestion around Hormuz/Gulf region, cargo throughput delays, 
    berth waiting times, container dwell time increases, port capacity constraints
  - 10_OtherIndustry: Petroleum/naphtha price changes, LNG spot price spikes, fertilizer price impacts, 
    bunkering price changes, petrochemical feedstock costs, refinery margin shifts"""

def build_classify_prompt(batch_articles, batch_contexts, mode='kg'):
    """분류 프롬프트 생성.
    mode='kg':  KG 엔티티 컨텍스트 활용 (KG Q1~Q7 수집 기사용)
    mode='mon': KG 없이 상세 카테고리 기준만으로 분류 (모니터링 D1~D9 수집 기사용)
    """
    items = []
    for j, (art, ctx) in enumerate(zip(batch_articles, batch_contexts)):
        title = art['title']
        lang = art.get('language', 'English')
        if mode == 'kg' and ctx:
            items.append(f"{j+1}. [{lang}] {title}\n   KG entities: {ctx}")
        else:
            items.append(f"{j+1}. [{lang}] {title}")

    news_block = '\n'.join(items)

    if mode == 'mon':
        # 모니터링: KG 없이 상세 relevance 기준 + 카테고리 분류
        prompt = f"""You are classifying news headlines for KMI (Korea Maritime Institute) daily maritime supply chain monitoring.

BACKGROUND — Korea's maritime supply chain vulnerabilities:
- Korea imports by sea: crude oil 95%, LNG 99%, naphtha 100%, iron ore 100%, grain 77%
- Key chokepoints: Hormuz (70% of Korea's oil), Malacca (85% energy transit), Suez/Bab el-Mandeb, Panama, Taiwan Strait
- Key dependencies: Middle East oil 71%, Qatar LNG 19% (via Hormuz), Australia LNG 25%, China urea 97%
- Key sectors affected: refining/petrochemical (naphtha-based), steel (iron ore/coal), shipping, food/agriculture
- Key Korean actors: SK Innovation, GS Caltex, S-Oil, POSCO, HMM, KOGAS, KEPCO, Samsung, Hyundai

STEP 1 — Relevance (use the criteria below strictly):
- HIGH: Article directly reports on:
  * Chokepoint disruption, blockade, military action, or transit restriction affecting Korea-bound routes
  * Korea's import supply disruption (oil, LNG, naphtha, coal, iron ore, grain, urea, semiconductors)
  * Korean port, shipping company, or energy infrastructure directly impacted
  * Korean government policy response to supply chain crisis (SPR release, emergency measures)
  * Freight rate surge or shipping reroute that directly affects Korea trade routes
  * Vessel seizure, attack, or accident on Korea-relevant shipping lanes

- MEDIUM: Article reports on:
  * Global chokepoint tension or military posturing (not yet directly disrupting Korea)
  * International oil/LNG/commodity price movements that will impact Korea's import costs
  * Major carrier (Maersk, MSC, CMA CGM) operational changes affecting Asia routes
  * IMO regulations, maritime safety developments with potential Korea impact
  * Financial market reactions (KOSPI, KRW, VIX) to maritime/energy events
  * Regional geopolitical developments (Iran, Houthi, US Navy) with supply chain implications
  * Seafood trade disruptions or fishing industry impacts from energy price changes

- LOW: Article is about:
  * Other countries' domestic supply chain issues with no clear Korea connection
  * General industry trends or corporate news without supply chain disruption angle
  * Historical analysis or opinion pieces without current operational impact
  * Maritime topics unrelated to supply chain (tourism, environment, sports)

- NONE: Article has no connection to maritime supply chain monitoring

STEP 2 — Category (assign the BEST matching ONE category for HIGH and MEDIUM; leave "" for LOW/NONE):
{CAT_BLOCK_MON}

Headlines:
{news_block}

Return ONLY valid JSON array:
[{{"index": 1, "relevance": "HIGH/MEDIUM/LOW/NONE", "topic": "2-3 word topic", "category": "1_Security or empty"}}]"""

    else:
        # KG: 엔티티 컨텍스트 활용
        prompt = f"""Classify each news headline's relevance to KOREAN maritime supply chain disruption risk.
Use the KG (Knowledge Graph) entity matches to inform your judgment.
Consider Korea's dependency on maritime imports (oil 95%, LNG 99%, naphtha 100%).

STEP 1 — Relevance:
- HIGH: Direct Korea supply chain impact (chokepoint blockage, commodity shortage, shipping disruption affecting Korea)
- MEDIUM: Indirect impact (freight rate changes, route congestion, energy price spikes, regional tension)
- LOW: Related but no immediate Korea impact (global trends, other-country specific, industry news)
- NONE: Not related to Korean maritime supply chain (even if KG entities appear in non-relevant context)

STEP 2 — Category (for HIGH and MEDIUM only, leave "" for LOW/NONE):
{CAT_BLOCK_KG}

Headlines with KG context:
{news_block}

Return ONLY valid JSON array:
[{{"index": 1, "relevance": "HIGH/MEDIUM/LOW/NONE", "topic": "2-3 word topic", "category": "1_Security or empty"}}]"""

    return prompt


def classify_group(classify_df, group_label, ckpt_file, mode='kg'):
    """한 그룹의 기사를 LLM으로 분류."""
    if len(classify_df) == 0:
        print(f"\n── {group_label}: 분류할 기사 없음 ──")
        return classify_df

    print(f"\n── {group_label}: LLM 분류 시작 ({len(classify_df)}건) ──")

    # 초기값
    classify_df = classify_df.copy()
    classify_df['relevance'] = 'NONE'
    classify_df['topic'] = ''
    classify_df['category'] = ''

    # 체크포인트 로드 (중단 재개)
    resume_batch = 0
    if os.path.exists(ckpt_file):
        ckpt = pd.read_csv(ckpt_file, encoding='utf-8-sig')
        ckpt_cols = list(ckpt.columns)
        has_cat = 'category' in ckpt_cols
        restored = 0
        for _, crow in ckpt.iterrows():
            mask = classify_df['url_hash'] == crow['url_hash']
            if mask.any():
                idx = classify_df[mask].index[0]
                classify_df.loc[idx, 'relevance'] = crow['relevance']
                classify_df.loc[idx, 'topic'] = crow['topic']
                if has_cat:
                    classify_df.loc[idx, 'category'] = crow.get('category', '')
                restored += 1
        resume_batch = restored // BATCH_SIZE
        print(f"  ⚡ 체크포인트: {restored}건 복원 → 배치 {resume_batch}부터 재개")
        del ckpt

    target_idx = classify_df.index.tolist()
    total = len(target_idx)
    total_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

    llm_calls = 0
    errors = 0
    t0 = time.time()

    for batch_num in range(total_batches):
        if batch_num < resume_batch:
            continue

        start = batch_num * BATCH_SIZE
        end = min(start + BATCH_SIZE, total)
        batch_idx = target_idx[start:end]

        batch_rows = classify_df.loc[batch_idx].to_dict('records')
        batch_contexts = classify_df.loc[batch_idx, '_kg_context'].tolist()

        prompt = build_classify_prompt(batch_rows, batch_contexts, mode=mode)
        result = call_llm_json(prompt, system=SYSTEM_MSG, max_tokens=16384)
        llm_calls += 1

        if result and isinstance(result, list):
            for item in result:
                j = item.get('index', 0) - 1
                if 0 <= j < len(batch_idx):
                    rel = item.get('relevance', 'LOW').upper()
                    if rel not in ('HIGH', 'MEDIUM', 'LOW', 'NONE'):
                        rel = 'LOW'
                    classify_df.loc[batch_idx[j], 'relevance'] = rel
                    classify_df.loc[batch_idx[j], 'topic'] = item.get('topic', '')
                    # 카테고리: HIGH/MEDIUM만, 유효한 키인지 검증
                    cat = item.get('category', '')
                    if rel in ('HIGH', 'MEDIUM') and cat in CATEGORY_KEYS:
                        classify_df.loc[batch_idx[j], 'category'] = cat
                    elif rel in ('HIGH', 'MEDIUM') and cat:
                        # 부분 매칭 시도 (LLM이 숫자만 반환하는 경우 등)
                        matched_cat = [k for k in CATEGORY_KEYS if cat in k or k.startswith(cat)]
                        if matched_cat:
                            classify_df.loc[batch_idx[j], 'category'] = matched_cat[0]
                        else:
                            classify_df.loc[batch_idx[j], 'category'] = cat  # 원본 유지
        else:
            errors += 1
            for idx in batch_idx:
                classify_df.loc[idx, 'relevance'] = 'LOW'
                classify_df.loc[idx, 'topic'] = 'classification_error'

        if (batch_num + 1) % 20 == 0 or batch_num == total_batches - 1:
            elapsed = time.time() - t0
            print(f"  [{batch_num+1}/{total_batches}] LLM {llm_calls}회, 오류 {errors}건 ({elapsed:.0f}초)")

        # 매 50배치마다 체크포인트
        if llm_calls > 0 and llm_calls % 50 == 0:
            _ckpt = classify_df[['url_hash', 'title', 'relevance', 'topic', 'category']].copy()
            _ckpt.to_csv(ckpt_file, index=False, encoding='utf-8-sig')
            del _ckpt
            print(f"  💾 체크포인트 저장 ({ckpt_file})")

        time.sleep(0.3)

    # 임시 필드 정리
    classify_df.drop(columns=['_kg_context'], inplace=True)

    # ── 분류 통계 ──
    elapsed = time.time() - t0
    rel_counts = classify_df['relevance'].value_counts()
    print(f"\n  [{group_label}] 분류 완료 ({elapsed:.0f}초, LLM {llm_calls}회)")
    for rel in ['HIGH', 'MEDIUM', 'LOW', 'NONE']:
        n = rel_counts.get(rel, 0)
        pct = n / len(classify_df) * 100
        print(f"    {rel:8s}: {n:5d}건 ({pct:.1f}%)")

    # KG 매칭별 분류 분포
    print(f"    KG 매칭 유무별:")
    for label, mask in [('매칭 1+', classify_df['kg_match_count'] > 0),
                         ('매칭 0', classify_df['kg_match_count'] == 0)]:
        sub = classify_df[mask]
        if len(sub) > 0:
            dist = sub['relevance'].value_counts()
            dist_str = ', '.join(f"{r}:{dist.get(r,0)}" for r in ['HIGH','MEDIUM','LOW','NONE'])
            print(f"      {label} ({len(sub)}건): {dist_str}")

    # 카테고리별 분포 (HIGH+MEDIUM)
    hm = classify_df[classify_df['relevance'].isin(['HIGH', 'MEDIUM'])]
    if len(hm) > 0:
        print(f"\n    카테고리별 분포 (HIGH+MEDIUM {len(hm)}건):")
        cat_counts = hm['category'].value_counts()
        for cat, cnt in cat_counts.items():
            label = cat if cat else '(미분류)'
            print(f"      {label:20s} {cnt:4d}건")

    return classify_df


# ── 실행 ──

print(f"\n✅ 공통 설정 완료. (함수: dedup_by_title / classify_group / call_llm_json 포함)")


=== GDELT 뉴스 수집기 ===
수집 언어: English
누적 파일: monitoring/cumulative/gdelt_mon_classified.csv
날짜 형식: YYYY-MM-DD  /  today  /  yesterday
Enter 치면 기본값이 적용됩니다.



📅 종료일 [2026-04-01]:  
📅 시작일 [2026-04-01]:  



수집 기간: 2026-04-01 ~ 2026-04-01 (1일)
  → 2026-04-01

✅ 공통 설정 완료. (함수: dedup_by_title / classify_group / call_llm_json 포함)


In [ ]:
# ============================================================
# Cell 2: 모니터링 차원 수집 (D1~D9) + 저장
# ============================================================
# news_queries_monitoring_v2.json → gdelt_mon_daily_*.csv
# 누적은 LLM 분류 후 gdelt_mon_classified.csv에서 (Cell 9)
# ============================================================

# ── 키워드 로드 ──
mon_keywords, mon_kw_source = load_keywords(MON_QUERY_FILE, lang='en')
print(f"=== 모니터링 차원 수집 (D1~D9) ===")
print(f"키워드: {len(mon_keywords)}개")

for qname in dict.fromkeys(mon_kw_source.values()):
    n = sum(1 for v in mon_kw_source.values() if v == qname)
    print(f"  {qname}: {n}개")

# ── 수집 대상 날짜 (이미 모니터링 분류된 날짜는 overwrite 아니면 제외) ──
mon_dates = []
for d in dates_to_collect:
    ds = str(d)
    if ds in already and d not in dates_overwrite:
        print(f"  건너뜀: {d} (MON 이미 분류됨)")
    else:
        mon_dates.append(d)

if not mon_dates:
    print("\n❌ 모니터링 수집할 날짜 없음")
    mon_df_new = pd.DataFrame()
else:
    est = len(mon_keywords) * len(mon_dates) * SLEEP_SEC / 60
    print(f"\n수집 시작: {len(mon_dates)}일 × {len(mon_keywords)}키워드 (~{est:.1f}분)\n")

    # ── 수집 ──
    mon_daily, mon_stats, mon_errors = collect_gdelt(mon_keywords, mon_kw_source, mon_dates)

    # ── daily CSV 저장 ──
    total = sum(len(v) for v in mon_daily.values())
    print(f"\n  수집 완료: {total}건")
    print(f"  저장 중...")
    mon_df_new = save_daily_csv(mon_daily, 'gdelt_mon')

    print(f"\n✅ 모니터링 수집 완료: {len(mon_df_new)}건")


=== 모니터링 차원 수집 (D1~D9) ===
키워드: 123개
  D1_초크포인트: 10개
  D2_품목: 26개
  D3_인프라: 8개
  D4_운송: 9개
  D5_시장지표: 16개
  D6_교란행위: 25개
  D7_행위자: 9개
  D8_수산물: 7개
  D9_한국: 13개

수집 시작: 1일 × 123키워드 (~1.0분)

  ⚠ GDELT 최소 길이(5자) 미달 → 스킵: ['corn']
    → news_queries_monitoring_v2.json에서 키워드를 길게 수정하세요.
  ── [1/1] 2026-03-30 (122키워드) ──
      ⚠ Suez Canal: 
      ⚠ Strait of Bab el-Mandeb: 
      ⚠ Strait of Malacca: 


In [28]:
# ============================================================
# Cell 3: 네이버 뉴스 수집 (Korean 전용)
# ============================================================
# 환경변수: NAVER_CLIENT_ID, NAVER_CLIENT_SECRET
# 입력:  news_queries_monitoring_v2.json (ko 키워드)
# 출력:  naver_df_new  (GDELT와 동일 컬럼 구조)
#        naver_mon_daily_YYYYMMDD.csv
# ============================================================

import requests, html, re as _re

# ── API 자격증명 ──
NAVER_CLIENT_ID     = os.environ.get('NAVER_CLIENT_ID', '')
NAVER_CLIENT_SECRET = os.environ.get('NAVER_CLIENT_SECRET', '')

if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    print("⚠ NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 환경변수가 설정되지 않았습니다.")
    print("  Jupyter 실행 전 터미널에서:")
    print("    export NAVER_CLIENT_ID=여기에_클라이언트ID")
    print("    export NAVER_CLIENT_SECRET=여기에_시크릿")
    print("  그 후 동일 터미널에서 jupyter notebook 실행")
    naver_df_new = pd.DataFrame()
else:
    NAVER_API_URL = 'https://openapi.naver.com/v1/search/news.json'
    NAVER_DISPLAY = 100   # 요청당 최대 100건
    NAVER_SLEEP   = 0.3   # API 호출 간격(초)

    def _clean_naver_text(text):
        """네이버 응답의 HTML 태그(<b>) 및 엔티티(&amp; 등) 제거."""
        text = _re.sub(r'<[^>]+>', '', text)
        return html.unescape(text).strip()

    def _parse_naver_date(pub_date_str):
        """RFC 822 날짜 문자열 → date 객체. 실패 시 None 반환."""
        try:
            from email.utils import parsedate_to_datetime
            return parsedate_to_datetime(pub_date_str).date()
        except Exception:
            return None

    def collect_naver(keywords, kw_source, target_dates, sleep=NAVER_SLEEP):
        """네이버 뉴스 API로 날짜별·키워드별 수집.
        - sort=date 정렬 후 pubDate 기반 필터링
        - 수집 범위(d_min~d_max)보다 오래된 기사 만나면 페이지네이션 중단
        """
        d_min = min(target_dates)
        d_max = max(target_dates)

        daily  = {d: [] for d in target_dates}
        stats  = {d: [] for d in target_dates}
        seen   = set()   # url_hash 중복 방지 (GDELT seen과 독립)

        headers = {
            'X-Naver-Client-Id':     NAVER_CLIENT_ID,
            'X-Naver-Client-Secret': NAVER_CLIENT_SECRET,
        }

        total_calls = 0
        for kw in keywords:
            new_by_date = {d: 0 for d in target_dates}
            start = 1

            while start <= 1000:   # 네이버 최대 1,000번째까지
                params = {
                    'query':   kw,
                    'display': NAVER_DISPLAY,
                    'start':   start,
                    'sort':    'date',
                }
                try:
                    resp = requests.get(NAVER_API_URL, params=params,
                                        headers=headers, timeout=10)
                    total_calls += 1

                    if resp.status_code == 429:
                        print(f"  ⚠ 요청 한도 초과 (429) — {kw}")
                        break
                    if resp.status_code != 200:
                        print(f"  ⚠ API 오류 {resp.status_code} — {kw}")
                        break

                    items = resp.json().get('items', [])
                    if not items:
                        break

                    oldest_in_batch = None
                    for item in items:
                        pub_date = _parse_naver_date(item.get('pubDate', ''))
                        if pub_date is None:
                            continue

                        if oldest_in_batch is None or pub_date < oldest_in_batch:
                            oldest_in_batch = pub_date

                        # 수집 범위 밖이면 저장 안 함
                        if not (d_min <= pub_date <= d_max):
                            continue

                        url = item.get('originallink', '') or item.get('link', '')
                        if not url:
                            continue

                        h = _make_hash(url)
                        if h in seen:
                            continue
                        seen.add(h)

                        title = _clean_naver_text(item.get('title', ''))
                        domain = url.split('/')[2] if url.startswith('http') else ''

                        daily[pub_date].append({
                            'url_hash':      h,
                            'title':         title,
                            'url':           url,
                            'seendate':      item.get('pubDate', ''),
                            'domain':        domain,
                            'language':      'Korean',
                            'sourcecountry': 'South Korea',
                            'query_keyword': kw,
                            'query_group':   kw_source.get(kw, ''),
                            'collect_date':  str(pub_date),
                        })
                        new_by_date[pub_date] += 1

                    # 배치 내 가장 오래된 기사가 수집 범위보다 이전이면 중단
                    if oldest_in_batch and oldest_in_batch < d_min:
                        break

                    start += NAVER_DISPLAY
                    time.sleep(sleep)

                except requests.exceptions.RequestException as e:
                    print(f"  ⚠ 요청 실패 ({kw}): {e}")
                    break

            for d, cnt in new_by_date.items():
                stats[d].append({'keyword': kw, 'new': cnt})

        print(f"  API 총 호출: {total_calls}회 (일별 한도 25,000)")
        return daily, stats

    # ── 키워드 로드 (모니터링 JSON, 한국어) ──
    naver_keywords, naver_kw_source = load_keywords(MON_QUERY_FILE, lang='ko')
    print(f"=== 네이버 뉴스 수집 ===")
    print(f"  키워드: {len(naver_keywords)}개")
    print(f"  수집 대상: {d_start} ~ {d_end} ({len(dates_to_collect)}일)")

    # ── 수집 실행 ──
    if not dates_to_collect:
        print("  ❌ 수집할 날짜 없음")
        naver_df_new = pd.DataFrame()
    else:
        est = len(naver_keywords) * len(dates_to_collect) * NAVER_SLEEP / 60
        print(f"  예상 소요: ~{est:.1f}분\n")

        naver_daily, naver_stats = collect_naver(naver_keywords, naver_kw_source, dates_to_collect)

        # ── daily CSV 저장 ──
        total = sum(len(v) for v in naver_daily.values())
        print(f"\n  수집 완료: {total}건")
        naver_df_new = save_daily_csv(naver_daily, 'naver_mon')

        print(f"\n✅ 네이버 수집 완료: {len(naver_df_new)}건")


=== 네이버 뉴스 수집 ===
  키워드: 89개
  수집 대상: 2026-03-29 ~ 2026-03-29 (1일)
  예상 소요: ~0.4분

  API 총 호출: 267회 (일별 한도 25,000)

  수집 완료: 3348건
    2026-03-29: 3348건 → monitoring/20260329/naver_mon_daily_20260329.csv

✅ 네이버 수집 완료: 3348건


In [38]:
# ============================================================
# Cell 4: 수집 통계 — GDELT + 네이버
# ============================================================

print(f"{'='*60}")
print(f"  모니터링 수집 리포트 — {d_start} ~ {d_end}")
print(f"{'='*60}")

# ── GDELT 통계 ──
if 'mon_df_new' not in dir() or len(mon_df_new) == 0:
    print("  [GDELT] 수집 0건")
else:
    all_errs = []
    for d in (mon_errors if 'mon_errors' in dir() else {}).values():
        all_errs.extend(d)
    print(f"  [GDELT] 신규 수집: {len(mon_df_new)}건, 에러 {len(all_errs)}건")

    grp = mon_df_new.groupby('query_group').size().sort_values(ascending=False)
    for g, n in grp.items():
        print(f"    {g:25s} {n:5d}건")

    kw_top = mon_df_new.groupby('query_keyword').size().nlargest(10)
    print(f"  상위 키워드 (Top 10):")
    for kw, n in kw_top.items():
        print(f"    {kw:35s} {n:4d}건")

    if 'mon_stats' in dir():
        all_stats_flat = []
        for d in mon_stats.values():
            all_stats_flat.extend(d)
        if all_stats_flat:
            df_s = pd.DataFrame(all_stats_flat)
            zero = df_s.groupby('keyword')['new'].sum()
            zero_kws = zero[zero == 0].index.tolist()
            if zero_kws:
                print(f"  0건 키워드: {len(zero_kws)}개 — {', '.join(zero_kws[:8])}" +
                      (f" 외 {len(zero_kws)-8}개" if len(zero_kws) > 8 else ""))

# ── 네이버 통계 ──
print()
if 'naver_df_new' not in dir() or len(naver_df_new) == 0:
    print("  [네이버] 수집 0건 (환경변수 미설정 또는 수집 없음)")
else:
    print(f"  [네이버] 신규 수집: {len(naver_df_new)}건")

    n_grp = naver_df_new.groupby('query_group').size().sort_values(ascending=False)
    for g, n in n_grp.items():
        print(f"    {g:25s} {n:5d}건")

    n_top = naver_df_new.groupby('query_keyword').size().nlargest(10)
    print(f"  상위 키워드 (Top 10):")
    for kw, n in n_top.items():
        print(f"    {kw:35s} {n:4d}건")

    if 'naver_stats' in dir():
        n_stats_flat = []
        for d in naver_stats.values():
            n_stats_flat.extend(d)
        if n_stats_flat:
            df_ns = pd.DataFrame(n_stats_flat)
            n_zero = df_ns.groupby('keyword')['new'].sum()
            n_zero_kws = n_zero[n_zero == 0].index.tolist()
            if n_zero_kws:
                print(f"  0건 키워드: {len(n_zero_kws)}개 — {', '.join(n_zero_kws[:8])}" +
                      (f" 외 {len(n_zero_kws)-8}개" if len(n_zero_kws) > 8 else ""))

# ── 합산 ──
gdelt_n  = len(mon_df_new)   if 'mon_df_new'   in dir() else 0
naver_n  = len(naver_df_new) if 'naver_df_new' in dir() else 0
print(f"\n  합계: GDELT {gdelt_n}건 + 네이버 {naver_n}건 = {gdelt_n + naver_n}건")

# ── 누적 현황 ──
if os.path.exists(CLASSIFIED_CSV):
    df_c = pd.read_csv(CLASSIFIED_CSV, encoding='utf-8-sig')
    n_dates = df_c['collect_date'].nunique()
    print(f"  누적 분류: {CLASSIFIED_CSV} ({len(df_c)}건, {n_dates}일)")
    del df_c

print(f"\n다음: Cell 5(GDELT dedup) → Cell 6(GDELT LLM) → Cell 7(네이버 dedup) → Cell 8(네이버 LLM) → Cell 9(저장) → Cell 10(기계적 리포트) → Cell 11(LLM 리포트)")


  모니터링 수집 리포트 — 2026-03-30 ~ 2026-03-30
  [GDELT] 신규 수집: 4381건, 에러 59건
    D2_품목                      1437건
    D3_인프라                      815건
    D4_운송                       695건
    D1_초크포인트                    508건
    D6_교란행위                     488건
    D9_한국                       164건
    D5_시장지표                     139건
    D7_행위자                       96건
    D8_수산물                       39건
  상위 키워드 (Top 10):
    Strait of Hormuz                     250건
    container                            242건
    delay                                237건
    carrier                              220건
    terminal                             214건
    natural gas                          206건
    petrochemical                        205건
    tanker                               201건
    fertilizer                           196건
    pipeline                             194건
  0건 키워드: 79개 — Baltic Dry Index, HMM shipping, Houthi, KOSPI, Korea construction, Korea economy, Korea energy, Korea

In [ ]:
# ============================================================
# Cell 5: GDELT 기사 중복 제거 + 비례 샘플링 → gdelt_classify_df
# ============================================================

MAX_LLM_SAMPLE_GDELT = 1500   # LLM에 넘길 최대 건수
MIN_PER_KEYWORD      = 3      # 키워드당 최소 보장 건수 (실제 기사 수보다 많으면 있는 것만)

def proportional_sample(df, max_n, min_per_kw=3, kw_col='query_keyword'):
    """키워드별 비례 샘플링.
    - 0건 키워드 제외
    - 전체 max_n건 한도 내에서 키워드별 기사 수 비율대로 배정
    - 키워드당 최소 min_per_kw건 보장 (단, 실제 기사가 부족하면 있는 것만)
    - 전체 건수가 max_n 이하이면 그대로 반환
    """
    if len(df) <= max_n:
        return df

    kw_counts = df[kw_col].value_counts()
    kw_counts = kw_counts[kw_counts > 0]
    total     = kw_counts.sum()

    # 비례 배정 (최솟값 보장 후 반올림)
    alloc = (kw_counts / total * max_n).clip(lower=min_per_kw).round().astype(int)

    # 반올림 오차 보정: 총합이 max_n이 되도록 (여러 키워드에 분산 처리)
    diff = max_n - alloc.sum()
    if diff > 0:
        # 부족분: 기사 여유 있는 키워드 순으로 채움
        for kw in kw_counts.index:
            if diff == 0:
                break
            spare = int(kw_counts[kw]) - alloc[kw]
            add   = min(spare, diff)
            if add > 0:
                alloc[kw] += add
                diff -= add
    elif diff < 0:
        # 초과분: 큰 키워드부터 최솟값 유지하며 감축
        excess = -diff
        for kw in alloc.index:  # value_counts() → 큰 순서 정렬됨
            if excess == 0:
                break
            reducible = alloc[kw] - min_per_kw
            reduce_by = min(reducible, excess)
            if reduce_by > 0:
                alloc[kw] -= reduce_by
                excess    -= reduce_by

    sampled = []
    for kw, n in alloc.items():
        sampled.append(df[df[kw_col] == kw].head(int(n)))
    return pd.concat(sampled, ignore_index=True)


gdelt_classify_df = pd.DataFrame()

# ── 메모리 변수 우선, 없으면 날짜별 폴더에서 fallback 로드 ──
if 'mon_df_new' in dir() and isinstance(mon_df_new, pd.DataFrame) and len(mon_df_new) > 0:
    _gdelt_src = mon_df_new
    print(f"  GDELT: 메모리에서 {len(_gdelt_src)}건 로드")
else:
    print("  ⚠ mon_df_new 없음 → 날짜별 폴더에서 CSV 로드 시도")
    _dfs = []
    for d in dates_to_collect:
        date_tag = d.strftime('%Y%m%d')
        csv_path = os.path.join(MONITOR_DIR, date_tag, f'gdelt_mon_daily_{date_tag}.csv')
        if os.path.exists(csv_path):
            _dfs.append(pd.read_csv(csv_path, encoding='utf-8-sig'))
            print(f"    {d}: {csv_path} 로드")
        else:
            print(f"    {d}: ❌ {csv_path} 없음")
    _gdelt_src = pd.concat(_dfs, ignore_index=True) if _dfs else pd.DataFrame()

if len(_gdelt_src) > 0:
    print(f"  GDELT: {len(_gdelt_src)}건 dedup 시작")
    _gdelt_dedup = dedup_by_title(_gdelt_src, 'GDELT')
    _gdelt_dedup['kg_entities']    = ''
    _gdelt_dedup['kg_match_count'] = 0
    _gdelt_dedup['_kg_context']    = ''
    print(f"  dedup 후: {len(_gdelt_dedup)}건")

    # ── 비례 샘플링 ──
    gdelt_classify_df = proportional_sample(
        _gdelt_dedup, max_n=MAX_LLM_SAMPLE_GDELT, min_per_kw=MIN_PER_KEYWORD
    )
    n_kw = gdelt_classify_df['query_keyword'].nunique()
    print(f"  샘플링 후: {len(_gdelt_dedup)}건 → {len(gdelt_classify_df)}건 ({n_kw}키워드)")
    print(f"\n✅ GDELT 분류 대상: {len(gdelt_classify_df)}건 → Cell 6에서 LLM 분류")
else:
    print("❌ GDELT 수집 기사 없음.")


In [36]:
# ============================================================
# Cell 6: GDELT LLM 분류 실행
# ============================================================
# (함수 정의는 Cell 1 공통 설정에 있음)

CLASSIFY_CKPT = os.path.join(CUMUL_DIR, 'gdelt_mon_classify_checkpoint.csv')

gdelt_classify_df = classify_group(gdelt_classify_df, 'GDELT 모니터링', CLASSIFY_CKPT, mode='mon')

if len(gdelt_classify_df) > 0:
    hm = gdelt_classify_df[gdelt_classify_df['relevance'].isin(['HIGH', 'MEDIUM'])]
    if len(hm) > 0:
        print(f"\n=== HIGH+MEDIUM 주요 토픽 ===")
        top_topics = hm['topic'].value_counts().head(10)
        for t, n in top_topics.items():
            print(f"  {t}: {n}건")

        print(f"\n=== 카테고리별 (HIGH+MEDIUM {len(hm)}건) ===")
        cat_all = hm['category'].value_counts()
        for cat, cnt in cat_all.items():
            label = cat if cat else '(미분류)'
            print(f"  {label:20s} {cnt:4d}건")

    print(f"\n✅ GDELT LLM 분류 완료 → Cell 7(네이버 dedup) → Cell 8(네이버 분류) → Cell 9(저장)")



── GDELT 모니터링: LLM 분류 시작 (3005건) ──
  [20/301] LLM 20회, 오류 0건 (59초)
  [40/301] LLM 40회, 오류 0건 (114초)
  💾 체크포인트 저장 (monitoring/cumulative/gdelt_mon_classify_checkpoint.csv)
  [60/301] LLM 60회, 오류 0건 (169초)
  [80/301] LLM 80회, 오류 0건 (228초)
  [100/301] LLM 100회, 오류 0건 (285초)
  💾 체크포인트 저장 (monitoring/cumulative/gdelt_mon_classify_checkpoint.csv)
  [120/301] LLM 120회, 오류 0건 (339초)
  [140/301] LLM 140회, 오류 0건 (395초)
  💾 체크포인트 저장 (monitoring/cumulative/gdelt_mon_classify_checkpoint.csv)
  [160/301] LLM 160회, 오류 0건 (451초)
  [180/301] LLM 180회, 오류 0건 (504초)
  [200/301] LLM 200회, 오류 0건 (563초)
  💾 체크포인트 저장 (monitoring/cumulative/gdelt_mon_classify_checkpoint.csv)
  [220/301] LLM 220회, 오류 0건 (621초)
  [240/301] LLM 240회, 오류 0건 (676초)
  💾 체크포인트 저장 (monitoring/cumulative/gdelt_mon_classify_checkpoint.csv)
  [260/301] LLM 260회, 오류 0건 (735초)
  [280/301] LLM 280회, 오류 0건 (797초)
  [300/301] LLM 300회, 오류 0건 (854초)
  💾 체크포인트 저장 (monitoring/cumulative/gdelt_mon_classify_checkpoint.csv)
  [301/301] LLM 301회,

In [ ]:
# ============================================================
# Cell 7: 네이버 기사 중복 제거 + 비례 샘플링 → naver_classify_df
# ============================================================

MAX_LLM_SAMPLE_NAVER = 1500   # LLM에 넘길 최대 건수 (proportional_sample은 Cell 5에서 정의)

naver_classify_df = pd.DataFrame()

# ── 메모리 변수 우선, 없으면 날짜별 폴더에서 fallback 로드 ──
if 'naver_df_new' in dir() and isinstance(naver_df_new, pd.DataFrame) and len(naver_df_new) > 0:
    _naver_src = naver_df_new
    print(f"  네이버: 메모리에서 {len(_naver_src)}건 로드")
else:
    print("  ⚠ naver_df_new 없음 → 날짜별 폴더에서 CSV 로드 시도")
    _dfs = []
    for d in dates_to_collect:
        date_tag = d.strftime('%Y%m%d')
        csv_path = os.path.join(MONITOR_DIR, date_tag, f'naver_mon_daily_{date_tag}.csv')
        if os.path.exists(csv_path):
            _dfs.append(pd.read_csv(csv_path, encoding='utf-8-sig'))
            print(f"    {d}: {csv_path} 로드")
        else:
            print(f"    {d}: ❌ {csv_path} 없음")
    _naver_src = pd.concat(_dfs, ignore_index=True) if _dfs else pd.DataFrame()

if len(_naver_src) > 0:
    print(f"  네이버: {len(_naver_src)}건 dedup 시작")
    _naver_dedup = dedup_by_title(_naver_src, '네이버')
    _naver_dedup['kg_entities']    = ''
    _naver_dedup['kg_match_count'] = 0
    _naver_dedup['_kg_context']    = ''
    print(f"  dedup 후: {len(_naver_dedup)}건")

    # ── 비례 샘플링 ──
    naver_classify_df = proportional_sample(
        _naver_dedup, max_n=MAX_LLM_SAMPLE_NAVER, min_per_kw=MIN_PER_KEYWORD
    )
    n_kw = naver_classify_df['query_keyword'].nunique()
    print(f"  샘플링 후: {len(_naver_dedup)}건 → {len(naver_classify_df)}건 ({n_kw}키워드)")
    print(f"\n✅ 네이버 분류 대상: {len(naver_classify_df)}건 → Cell 8에서 LLM 분류")
else:
    print("❌ 네이버 수집 기사 없음 (환경변수 미설정 또는 수집 없음).")


In [39]:
# ============================================================
# Cell 8: 네이버 LLM 분류 (HIGH/MEDIUM/LOW/NONE + 카테고리)
# ============================================================
# classify_group() / call_llm_json() / CATEGORIES 등은 Cell 1(공통 설정)에서 정의됨

NAVER_CLASSIFY_CKPT = os.path.join(CUMUL_DIR, 'naver_mon_classify_checkpoint.csv')

naver_classify_df = classify_group(naver_classify_df, '네이버 모니터링', NAVER_CLASSIFY_CKPT, mode='mon')

if len(naver_classify_df) > 0:
    hm = naver_classify_df[naver_classify_df['relevance'].isin(['HIGH', 'MEDIUM'])]
    if len(hm) > 0:
        print(f"\n=== HIGH+MEDIUM 주요 토픽 ===")
        top_topics = hm['topic'].value_counts().head(10)
        for t, n in top_topics.items():
            print(f"  {t}: {n}건")

        print(f"\n=== 카테고리별 (HIGH+MEDIUM {len(hm)}건) ===")
        cat_all = hm['category'].value_counts()
        for cat, cnt in cat_all.items():
            label = cat if cat else '(미분류)'
            print(f"  {label:20s} {cnt:4d}건")

    print(f"\n✅ 네이버 LLM 분류 완료 → Cell 9에서 저장")



── 네이버 모니터링: LLM 분류 시작 (8180건) ──
  [20/818] LLM 20회, 오류 0건 (67초)
  [40/818] LLM 40회, 오류 0건 (127초)
  💾 체크포인트 저장 (monitoring/cumulative/naver_mon_classify_checkpoint.csv)
  [60/818] LLM 60회, 오류 0건 (189초)
  [80/818] LLM 80회, 오류 0건 (248초)
  [100/818] LLM 100회, 오류 0건 (309초)
  💾 체크포인트 저장 (monitoring/cumulative/naver_mon_classify_checkpoint.csv)
  [120/818] LLM 120회, 오류 0건 (372초)
  [140/818] LLM 140회, 오류 0건 (432초)
  💾 체크포인트 저장 (monitoring/cumulative/naver_mon_classify_checkpoint.csv)
  [160/818] LLM 160회, 오류 0건 (502초)
  [180/818] LLM 180회, 오류 0건 (562초)
  [200/818] LLM 200회, 오류 0건 (624초)
  💾 체크포인트 저장 (monitoring/cumulative/naver_mon_classify_checkpoint.csv)
  [220/818] LLM 220회, 오류 0건 (687초)
  [240/818] LLM 240회, 오류 0건 (750초)
  💾 체크포인트 저장 (monitoring/cumulative/naver_mon_classify_checkpoint.csv)
  [260/818] LLM 260회, 오류 0건 (811초)
  [280/818] LLM 280회, 오류 0건 (875초)
  [300/818] LLM 300회, 오류 0건 (932초)
  💾 체크포인트 저장 (monitoring/cumulative/naver_mon_classify_checkpoint.csv)
  [320/818] LLM 320회, 오

In [ ]:
# ============================================================
# Cell 9: 분류 결과 저장 — GDELT + 네이버 각각 저장
# (주간 롤링은 Cell 10에서 별도 처리)
# ============================================================

# NAVER_CLASSIFY_CKPT는 Cell 8에서 정의됨

def save_classified(classify_df, classified_csv, daily_prefix, ckpt_file):
    """분류된 기사를 당일 CSV + 누적 CSV에 저장."""
    if len(classify_df) == 0:
        print("저장할 분류 결과 없음")
        return

    save_cols = [
        'url_hash', 'title', 'url', 'seendate', 'domain', 'language',
        'sourcecountry', 'query_keyword', 'query_group', 'collect_date',
        'kg_entities', 'kg_match_count', 'relevance', 'topic', 'category'
    ]
    save_cols = [c for c in save_cols if c in classify_df.columns]
    df_save = classify_df[save_cols].copy()

    rel_order = {'HIGH': 0, 'MEDIUM': 1, 'LOW': 2, 'NONE': 3}
    df_save['_rel_order'] = df_save['relevance'].map(rel_order)
    df_save = df_save.sort_values(['_rel_order', 'seendate'], ascending=[True, False])
    df_save = df_save.drop(columns=['_rel_order']).reset_index(drop=True)

    # ── 당일 CSV (날짜별 폴더에 저장) ──
    for cd, grp in df_save.groupby('collect_date'):
        date_tag = cd.replace('-', '')
        daily_dir = os.path.join(MONITOR_DIR, date_tag)
        os.makedirs(daily_dir, exist_ok=True)
        daily_csv = os.path.join(daily_dir, f'{daily_prefix}_daily_{date_tag}.csv')
        grp.to_csv(daily_csv, index=False, encoding='utf-8-sig')
        print(f"  당일: {daily_csv} ({len(grp)}건)")

    # ── 누적 CSV 병합 ──
    if os.path.exists(classified_csv):
        df_existing = pd.read_csv(classified_csv, encoding='utf-8-sig')
        overlap_dates = set(df_save['collect_date'].unique())
        df_keep = df_existing[~df_existing['collect_date'].isin(overlap_dates)]
        df_final = pd.concat([df_keep, df_save], ignore_index=True)
        print(f"  누적: 기존 {len(df_keep)}건 + 신규 {len(df_save)}건 = {len(df_final)}건")
    else:
        df_final = df_save.copy()
        print(f"  누적: 첫 파일 생성 ({len(df_final)}건)")

    df_final.to_csv(classified_csv, index=False, encoding='utf-8-sig')
    print(f"  → {classified_csv} ({os.path.getsize(classified_csv)//1024} KB)")

    rel_counts = df_save['relevance'].value_counts()
    high_n = rel_counts.get('HIGH', 0)
    med_n  = rel_counts.get('MEDIUM', 0)
    print(f"  분류: {len(df_save)}건 (HIGH:{high_n}, MEDIUM:{med_n})")

    if high_n > 0:
        print(f"\n  🔴 HIGH 기사:")
        for _, row in df_save[df_save['relevance'] == 'HIGH'].head(10).iterrows():
            cat_tag = f"[{row.get('category','')}] " if row.get('category','') else ''
            print(f"    {cat_tag}[{row.get('topic','')}] {row['title'][:80]}")

    if os.path.exists(ckpt_file):
        os.remove(ckpt_file)
        print(f"  🗑 체크포인트 삭제")


# ── GDELT 저장 ──
print("\n── GDELT 저장 ──")
if 'gdelt_classify_df' in dir() and len(gdelt_classify_df) > 0:
    save_classified(gdelt_classify_df, CLASSIFIED_CSV, 'gdelt_mon_classified', CLASSIFY_CKPT)
else:
    print("  저장할 GDELT 분류 결과 없음")

# ── 네이버 저장 ──
print("\n── 네이버 저장 ──")
if 'naver_classify_df' in dir() and len(naver_classify_df) > 0:
    save_classified(naver_classify_df, NAVER_CLASSIFIED_CSV, 'naver_mon_classified', NAVER_CLASSIFY_CKPT)
else:
    print("  저장할 네이버 분류 결과 없음")

# ── 합산 요약 ──
print(f"\n{'='*60}")
print(f"  수집·분류 완료 — {d_start} ~ {d_end}")
print(f"{'='*60}")

# ── 체크포인트 삭제 ──
for ckpt in [CLASSIFY_CKPT, NAVER_CLASSIFY_CKPT]:
    if os.path.exists(ckpt):
        os.remove(ckpt)
        print(f'  🗑 {os.path.basename(ckpt)} 삭제')

for label, csv_path in [('GDELT', CLASSIFIED_CSV), ('네이버', NAVER_CLASSIFIED_CSV)]:
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path, encoding='utf-8-sig')
        n_dates = df['collect_date'].nunique()
        rel = df['relevance'].value_counts()
        print(f"  [{label}] 누적: {len(df)}건, {n_dates}일")
        print(f"    HIGH:{rel.get('HIGH',0)} MEDIUM:{rel.get('MEDIUM',0)} LOW:{rel.get('LOW',0)} NONE:{rel.get('NONE',0)}")
        del df

print(f"\n✅ 저장 완료 → Cell 10에서 주간 롤링 재구성, Cell 11에서 데일리 리포트 생성")


In [45]:
# ============================================================
# Cell 10: 주간 롤링 파일 재구성 — 독립 실행 가능
# ─────────────────────────────────────────────────────────
# - 메모리 상태 불필요 (수집/LLM 분류 재실행 없이 단독 실행 가능)
# - monitoring/YYYYMMDD/ 폴더에서 daily CSV를 스캔하여 주간 파일 재구성
# - 같은 날짜가 이미 포함된 경우 교체 → 멱등 (재실행해도 중복 없음)
# - 파일명 기준: 해당 날짜가 속한 주의 일요일 날짜
#   예) 3/30(월) → week_20260405.csv, 3/29(일) → week_20260329.csv
# ============================================================

import glob
from datetime import timedelta

WEEKLY_DIR = os.path.join(MONITOR_DIR, 'weekly')
os.makedirs(WEEKLY_DIR, exist_ok=True)

for daily_prefix in ['gdelt_mon_classified', 'naver_mon_classified']:
    print(f"\n── {daily_prefix} 주간 롤링 재구성 ──")

    # monitoring/YYYYMMDD/{prefix}_daily_YYYYMMDD.csv 전체 스캔
    pattern = os.path.join(MONITOR_DIR, '*', f'{daily_prefix}_daily_*.csv')
    daily_files = sorted(glob.glob(pattern))
    if not daily_files:
        print(f"  daily CSV 없음, 건너뜀")
        continue

    # 날짜별 → 주별 그룹화
    week_map = {}  # sunday_tag → [DataFrame, ...]
    for f in daily_files:
        df = pd.read_csv(f, encoding='utf-8-sig')
        if df.empty or 'collect_date' not in df.columns:
            continue
        for cd, grp in df.groupby('collect_date'):
            cd_date = pd.to_datetime(cd).date()
            sunday  = cd_date + timedelta(days=(6 - cd_date.weekday()))
            sunday_tag = sunday.strftime('%Y%m%d')
            if sunday_tag not in week_map:
                week_map[sunday_tag] = {}
            week_map[sunday_tag][cd] = grp  # 날짜별 최신 데이터만 유지

    # 주별로 합산 후 저장
    for sunday_tag in sorted(week_map.keys()):
        frames = list(week_map[sunday_tag].values())
        wk_df  = pd.concat(frames, ignore_index=True)
        weekly_csv = os.path.join(WEEKLY_DIR, f'{daily_prefix}_week_{sunday_tag}.csv')
        wk_df.to_csv(weekly_csv, index=False, encoding='utf-8-sig')
        dates_in = sorted(week_map[sunday_tag].keys())
        print(f"  week_{sunday_tag}: {len(wk_df)}건 ({len(dates_in)}일치: {', '.join(dates_in)})")

print(f"\n✅ 주간 롤링 재구성 완료 → {WEEKLY_DIR}")



── gdelt_mon_classified 주간 롤링 재구성 ──
  week_20260329: 10883건 (4일치: 2026-03-26, 2026-03-27, 2026-03-28, 2026-03-29)
  week_20260405: 3005건 (1일치: 2026-03-30)

── naver_mon_classified 주간 롤링 재구성 ──
  week_20260329: 18791건 (4일치: 2026-03-26, 2026-03-27, 2026-03-28, 2026-03-29)
  week_20260405: 8180건 (1일치: 2026-03-30)

✅ 주간 롤링 재구성 완료 → monitoring/weekly


In [41]:
# ============================================================
# Cell 10: 통계 Excel 리포트 자동 생성
# ============================================================
# 입력:  gdelt_mon_classified_daily_YYYYMMDD.csv + naver_mon_classified_daily_YYYYMMDD.csv
# 출력:  daily_report_YYYYMMDD.xlsx
# ============================================================

import re

# ── 카테고리 한글명 매핑 ──
CAT_KR = {
    "1_Security":       "안보·군사",
    "2_Safety":         "해양안전",
    "3_Freight":        "운임·물류비",
    "4_PortCargo":      "항만·화물",
    "5_EconFinance":    "경제·금융",
    "6_Seafood":        "수산물",
    "7_Shipping":       "해운",
    "8_Logistics":      "물류",
    "9_PortCongestion": "항만혼잡",
    "10_OtherIndustry": "기타 산업",
}
CAT_ORDER = list(CAT_KR.keys())

# ── 키워드 추적 목록 ──
TRACKED_KEYWORDS = [
    ('호르무즈|Hormuz',                '호르무즈'),
    ('이란|Iran|IRGC',                 '이란'),
    ('트럼프|Trump',                   '트럼프'),
    ('나프타|naphtha|납사',             '나프타'),
    ('LNG',                            'LNG'),
    ('원유|crude oil|petroleum',       '원유'),
    ('유가|oil price',                 '유가'),
    ('코스피|KOSPI',                   '코스피'),
    ('환율|exchange rate|won-dollar',  '환율'),
    ('수에즈|Suez',                    '수에즈'),
    ('말라카|Malacca',                 '말라카'),
    ('운임|freight rate|tanker rate',  '운임'),
    ('셧다운|shutdown',                '셧다운'),
    ('비상경영|emergency',              '비상경영'),
    ('OECD',                           'OECD'),
    ('반도체|semiconductor',           '반도체'),
    ('석유화학|petrochemical',          '석유화학'),
    ('물가|inflation|CPI',             '물가'),
    ('홍해|Red Sea|Houthi',           '홍해'),
    ('해운|shipping|carrier',          '해운'),
]


# ══════════════════════════════════════════════════════════════
# 공통 데이터 추출
# ══════════════════════════════════════════════════════════════

def _extract_report_data(target_date=None):
    """분류 CSV에서 리포트에 필요한 모든 데이터를 추출. dict 반환."""

    if target_date is None:
        target_date = d_end
    td = str(target_date).replace('-', '')
    td_fmt = str(target_date)

    # CSV 로드
    daily_dir = os.path.join(MONITOR_DIR, td)
    gdelt_csv = os.path.join(daily_dir, f'gdelt_mon_classified_daily_{td}.csv')
    naver_csv = os.path.join(daily_dir, f'naver_mon_classified_daily_{td}.csv')

    dfs = []
    sources_loaded = []
    for csv_path, label in [(gdelt_csv, 'GDELT'), (naver_csv, '네이버')]:
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path, encoding='utf-8-sig')
            df['_source'] = label
            dfs.append(df)
            sources_loaded.append(f"{label}({len(df)})")
        else:
            sources_loaded.append(f"{label}(없음)")

    if not dfs:
        print("❌ 분류 CSV가 없습니다. Cell 9까지 먼저 실행하세요.")
        return None

    print(f"📂 로드: {', '.join(sources_loaded)}")

    df_all = pd.concat(dfs, ignore_index=True)
    df_all = df_all.drop_duplicates(subset='url_hash', keep='first').reset_index(drop=True)

    total = len(df_all)
    rel_counts = df_all['relevance'].value_counts()
    n_high = rel_counts.get('HIGH', 0)
    n_med  = rel_counts.get('MEDIUM', 0)
    n_low  = rel_counts.get('LOW', 0)
    n_none = rel_counts.get('NONE', 0)

    df_all['source_type'] = df_all['sourcecountry'].apply(
        lambda x: 'domestic' if x == 'South Korea' else 'international'
    )

    hm = df_all[df_all['relevance'].isin(['HIGH', 'MEDIUM'])].copy()
    hm_kr = hm[hm['language'] == 'Korean']
    hm_domestic = hm[hm['source_type'] == 'domestic']
    hm_intl     = hm[hm['source_type'] == 'international']

    # 카테고리별 통계
    cat_stats = {}
    for cat in CAT_ORDER:
        cat_hm = hm[hm['category'] == cat]
        h = len(cat_hm[cat_hm['relevance'] == 'HIGH'])
        m = len(cat_hm[cat_hm['relevance'] == 'MEDIUM'])
        t = h + m
        pct = t / len(hm) * 100 if len(hm) > 0 else 0
        cat_stats[cat] = {'high': h, 'med': m, 'total': t, 'pct': pct}

    # 키워드 빈도
    kw_results = []
    for pattern, label in TRACKED_KEYWORDS:
        count = len(hm[hm['title'].str.contains(pattern, case=False, na=False)])
        if count > 0:
            kw_results.append((label, count, count / len(hm) * 100 if len(hm) > 0 else 0))
    kw_results.sort(key=lambda x: -x[1])

    # 전일 데이터
    prev_date = (pd.Timestamp(td_fmt) - pd.Timedelta(days=1)).strftime('%Y%m%d')
    prev_data = None
    _prev_dfs = []
    _prev_dir = os.path.join(MONITOR_DIR, prev_date)
    for _pcsv in [os.path.join(_prev_dir, f'gdelt_mon_classified_daily_{prev_date}.csv'),
                  os.path.join(_prev_dir, f'naver_mon_classified_daily_{prev_date}.csv')]:
        if os.path.exists(_pcsv):
            _prev_dfs.append(pd.read_csv(_pcsv, encoding='utf-8-sig'))
    if _prev_dfs:
        df_prev = pd.concat(_prev_dfs, ignore_index=True)
        hm_prev = df_prev[df_prev['relevance'].isin(['HIGH', 'MEDIUM'])]
        prev_cat = {}
        for cat in CAT_ORDER:
            prev_cat[cat] = len(hm_prev[hm_prev['category'] == cat])
        prev_data = {
            'date': f"{prev_date[:4]}-{prev_date[4:6]}-{prev_date[6:]}",
            'total': len(df_prev),
            'hm_total': len(hm_prev),
            'cat': prev_cat,
        }

    return {
        'td': td, 'td_fmt': td_fmt,
        'total': total, 'n_high': n_high, 'n_med': n_med,
        'n_low': n_low, 'n_none': n_none,
        'hm': hm, 'hm_kr': hm_kr,
        'hm_domestic': hm_domestic, 'hm_intl': hm_intl,
        'cat_stats': cat_stats,
        'kw_results': kw_results,
        'prev_data': prev_data,
        'df_all': df_all,
    }


# ══════════════════════════════════════════════════════════════
# 1) Markdown 리포트
# ══════════════════════════════════════════════════════════════

def _generate_xlsx(data):
    """openpyxl로 다중 시트 Excel 리포트 생성."""
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side, numbers
    from openpyxl.utils import get_column_letter

    td_fmt = data['td_fmt']
    hm = data['hm']
    hm_kr = data['hm_kr']
    hm_domestic, hm_intl = data['hm_domestic'], data['hm_intl']
    cat_stats = data['cat_stats']

    wb = Workbook()

    # 공통 스타일
    header_font = Font(name='Malgun Gothic', bold=True, size=11, color='FFFFFF')
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    high_fill   = PatternFill(start_color='FCE4EC', end_color='FCE4EC', fill_type='solid')
    med_fill    = PatternFill(start_color='FFF8E1', end_color='FFF8E1', fill_type='solid')
    cell_font   = Font(name='Malgun Gothic', size=10)
    thin_border = Border(
        left=Side(style='thin', color='CCCCCC'),
        right=Side(style='thin', color='CCCCCC'),
        top=Side(style='thin', color='CCCCCC'),
        bottom=Side(style='thin', color='CCCCCC'),
    )

    def _style_header(ws, row=1, max_col=None):
        if max_col is None:
            max_col = ws.max_column
        for col in range(1, max_col + 1):
            cell = ws.cell(row=row, column=col)
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.border = thin_border

    def _auto_width(ws, min_width=8, max_width=60):
        for col_cells in ws.columns:
            max_len = 0
            col_letter = get_column_letter(col_cells[0].column)
            for cell in col_cells:
                if cell.value:
                    # 한글은 대략 2배 너비
                    val = str(cell.value)
                    kr_chars = sum(1 for c in val if ord(c) > 0x1100)
                    max_len = max(max_len, len(val) + kr_chars)
            ws.column_dimensions[col_letter].width = min(max(max_len + 2, min_width), max_width)

    # ── Sheet 1: 요약 (Summary) ──
    ws1 = wb.active
    ws1.title = '요약'

    ws1.cell(1, 1, f'해상 공급망 모니터링 일일 요약 — {td_fmt}').font = Font(name='Malgun Gothic', bold=True, size=14)
    ws1.merge_cells('A1:E1')

    # 통계
    ws1.cell(3, 1, '지표').font = Font(bold=True)
    ws1.cell(3, 2, '값').font = Font(bold=True)
    stats_rows = [
        ('총 수집', data['total']),
        ('HIGH', data['n_high']),
        ('MEDIUM', data['n_med']),
        ('LOW', data['n_low']),
        ('NONE', data['n_none']),
        ('HIGH+MEDIUM', len(hm)),
        ('한국어 (H+M)', len(hm_kr)),
    ]
    for i, (k, v) in enumerate(stats_rows, 4):
        ws1.cell(i, 1, k).font = cell_font
        ws1.cell(i, 2, v).font = cell_font

    # 카테고리 분포
    cat_start = len(stats_rows) + 6
    ws1.cell(cat_start, 1, '카테고리별 분포').font = Font(name='Malgun Gothic', bold=True, size=12)
    ws1.merge_cells(f'A{cat_start}:E{cat_start}')

    h_row = cat_start + 1
    for j, h in enumerate(['카테고리', 'HIGH', 'MEDIUM', '합계', '비중(%)'], 1):
        ws1.cell(h_row, j, h)
    _style_header(ws1, row=h_row, max_col=5)

    for i, cat in enumerate(CAT_ORDER, h_row + 1):
        s = cat_stats[cat]
        ws1.cell(i, 1, CAT_KR[cat]).font = cell_font
        ws1.cell(i, 2, s['high']).font = cell_font
        ws1.cell(i, 3, s['med']).font = cell_font
        ws1.cell(i, 4, s['total']).font = cell_font
        ws1.cell(i, 5, round(s['pct'], 1)).font = cell_font
        for col in range(1, 6):
            ws1.cell(i, col).border = thin_border

    _auto_width(ws1)

    # ── Sheet 2: 전체 기사 목록 (HIGH+MEDIUM) ──
    ws2 = wb.create_sheet('기사목록_HIGH_MED')
    cols = ['relevance', 'category', 'language', 'title', 'topic', 'domain', 'url']
    col_kr = ['관련도', '카테고리', '언어', '제목', '토픽', '도메인', 'URL']
    for j, h in enumerate(col_kr, 1):
        ws2.cell(1, j, h)
    _style_header(ws2, max_col=len(cols))

    for i, (_, row) in enumerate(hm.iterrows(), 2):
        for j, col in enumerate(cols, 1):
            val = row.get(col, '')
            ws2.cell(i, j, val if pd.notna(val) else '').font = cell_font
            ws2.cell(i, j).border = thin_border
        # 조건부 색상
        if row.get('relevance') == 'HIGH':
            for j in range(1, len(cols)+1):
                ws2.cell(i, j).fill = high_fill
        elif row.get('relevance') == 'MEDIUM':
            for j in range(1, len(cols)+1):
                ws2.cell(i, j).fill = med_fill

    # 자동 필터
    ws2.auto_filter.ref = f'A1:{get_column_letter(len(cols))}{len(hm)+1}'
    _auto_width(ws2)

    # ── Sheet 3: 국내 기사 ──
    ws3 = wb.create_sheet('국내기사')
    for j, h in enumerate(col_kr, 1):
        ws3.cell(1, j, h)
    _style_header(ws3, max_col=len(cols))

    for i, (_, row) in enumerate(hm_domestic.iterrows(), 2):
        for j, col in enumerate(cols, 1):
            val = row.get(col, '')
            ws3.cell(i, j, val if pd.notna(val) else '').font = cell_font
            ws3.cell(i, j).border = thin_border
        if row.get('relevance') == 'HIGH':
            for j in range(1, len(cols)+1):
                ws3.cell(i, j).fill = high_fill

    ws3.auto_filter.ref = f'A1:{get_column_letter(len(cols))}{len(hm_domestic)+1}'
    _auto_width(ws3)

    # ── Sheet 4: 키워드 빈도 ──
    ws4 = wb.create_sheet('키워드빈도')
    for j, h in enumerate(['키워드', '언급 건수', '비중(%)'], 1):
        ws4.cell(1, j, h)
    _style_header(ws4, max_col=3)

    for i, (label, count, pct) in enumerate(data['kw_results'], 2):
        ws4.cell(i, 1, label).font = cell_font
        ws4.cell(i, 2, count).font = cell_font
        ws4.cell(i, 3, round(pct, 1)).font = cell_font
        for j in range(1, 4):
            ws4.cell(i, j).border = thin_border

    _auto_width(ws4)

    # ── Sheet 5: 전일 대비 (있을 경우) ──
    prev = data['prev_data']
    if prev:
        ws5 = wb.create_sheet('전일대비')
        for j, h in enumerate(['카테고리', f"전일({prev['date']})", f"당일({td_fmt})", '변화', '방향'], 1):
            ws5.cell(1, j, h)
        _style_header(ws5, max_col=5)

        row_idx = 2
        for cat in CAT_ORDER:
            p = prev['cat'][cat]; c = cat_stats[cat]['total']
            if p > 0 or c > 0:
                diff = c - p
                arrow = "↑" if diff > 0 else ("↓" if diff < 0 else "→")
                ws5.cell(row_idx, 1, CAT_KR[cat]).font = cell_font
                ws5.cell(row_idx, 2, p).font = cell_font
                ws5.cell(row_idx, 3, c).font = cell_font
                ws5.cell(row_idx, 4, diff).font = cell_font
                ws5.cell(row_idx, 5, arrow).font = cell_font
                for j in range(1, 6):
                    ws5.cell(row_idx, j).border = thin_border
                row_idx += 1

        _auto_width(ws5)

    out_path = os.path.join(MONITOR_DIR, data['td'], f'daily_report_{data["td"]}.xlsx')
    wb.save(out_path)
    return out_path


# ══════════════════════════════════════════════════════════════
# 통합 실행
# ══════════════════════════════════════════════════════════════

def generate_daily_report(target_date=None):
    """통계 xlsx만 생성."""
    data = _extract_report_data(target_date)
    if data is None:
        return

    print(f"\n{'='*50}")
    print(f"  📊 통계 리포트 생성 — {data['td_fmt']}")
    print(f"{'='*50}")

    # Excel
    xlsx_path = None
    try:
        xlsx_path = _generate_xlsx(data)
        print(f"  ✅ {xlsx_path}")
    except ImportError:
        print("  ⚠  openpyxl 미설치 → pip install openpyxl")
    except Exception as e:
        print(f"  ❌ xlsx 생성 실패: {e}")

    print(f"\n  HIGH+MEDIUM {len(data['hm']):,}건, "
          f"카테고리 {len([c for c in data['cat_stats'] if data['cat_stats'][c]['total']>0])}개")

    return {'xlsx': xlsx_path, 'data': data}


# ── 실행: dates_to_collect 전체 날짜 리포트 생성 ──
results = {}
for _report_date in dates_to_collect:
    print(f"\n{'='*60}")
    print(f"  📅 {_report_date} 리포트 생성 중...")
    results[str(_report_date)] = generate_daily_report(_report_date)

print(f"\n{'='*60}")
print(f"✅ 총 {len(results)}일 리포트 생성 완료")
for _d, _r in results.items():
    if _r:
        print(f"  {_d}: xlsx")
    else:
        print(f"  {_d}: ❌ 생성 실패 (분류 CSV 없음)")


  📅 2026-03-30 리포트 생성 중...
📂 로드: GDELT(3005), 네이버(8180)

  📊 통계 리포트 생성 — 2026-03-30
  ✅ monitoring/20260330/daily_report_20260330.xlsx

  HIGH+MEDIUM 3,593건, 카테고리 10개

✅ 총 1일 리포트 생성 완료
  2026-03-30: xlsx


In [42]:
# ============================================================
# Cell 11: LLM 데일리 리포트 생성 (Claude Sonnet)
# ============================================================
# 입력: gdelt_mon_classified_daily_*.csv + naver_mon_classified_daily_*.csv
# 출력:
#   daily_report_llm_YYYYMMDD.md   — 기존 10카테고리 마크다운
#   daily_report_llm_YYYYMMDD.docx — 기존 10카테고리 docx
#   daily_report_html_YYYYMMDD.html — 신규 공급망 흐름 HTML
# LLM 호출: 1회 (categories + flow + changes 통합 JSON)
# ── Cell 10의 _extract_report_data(), CAT_KR, CAT_ORDER 사용 ──
# ============================================================

import anthropic, json as _json, os
from datetime import datetime

LLM_REPORT_MODEL = "claude-sonnet-4-6"

# ── CAT_KR / CAT_ORDER 미정의 시 fallback ──
if 'CAT_KR' not in dir():
    CAT_KR = {
        "1_Security":       "안보·군사",
        "2_Safety":         "해양안전",
        "3_Freight":        "운임·물류비",
        "4_PortCargo":      "항만·화물",
        "5_EconFinance":    "경제·금융",
        "6_Seafood":        "수산물",
        "7_Shipping":       "해운",
        "8_Logistics":      "물류",
        "9_PortCongestion": "항만혼잡",
        "10_OtherIndustry": "기타 산업",
    }
    CAT_ORDER = list(CAT_KR.keys())

SYSTEM_PROMPT = """당신은 KMI(한국해양수산개발원) 해상 공급망 위기 모니터링 시스템의 전문 분석가입니다.
현재 핵심 사태: 2026년 호르무즈 해협 봉쇄 위기 (역사상 최초 실제 봉쇄)
분석 관점: 글로벌 해상 공급망 교란 → 한국 경제·산업·소비자에 미치는 영향
원칙: 기사 제목 사실 기반으로만 서술. 과장·추론 금지. 불확실하면 '~로 보임', '~가능성' 명시.
출력: 반드시 유효한 JSON만 출력. 설명 텍스트나 마크다운 코드블록 없이 JSON 객체만."""


def _build_report_prompt(data):
    """LLM 호출용 통합 프롬프트 (categories + flow + changes 한 번에)"""
    td_fmt   = data['td_fmt']
    hm       = data['hm']
    hm_intl  = data['hm_intl']
    hm_dom   = data['hm_domestic']

    # ── 카테고리별 기사 블록 ──
    active_cats   = []
    articles_block = ""
    for cat in CAT_ORDER:
        cat_hm = hm[hm['category'] == cat]
        if len(cat_hm) == 0:
            continue
        active_cats.append(cat)
        intl_rows = cat_hm[cat_hm['source_type'] == 'international'].head(10)
        dom_rows  = cat_hm[cat_hm['source_type'] == 'domestic'].head(10)
        h = len(cat_hm[cat_hm['relevance'] == 'HIGH'])
        m = len(cat_hm[cat_hm['relevance'] == 'MEDIUM'])
        articles_block += f"\n[{CAT_KR[cat]}] HIGH:{h} MEDIUM:{m}\n"
        if len(intl_rows) > 0:
            articles_block += f"  해외({len(intl_rows)}건):\n"
            for _, r in intl_rows.iterrows():
                articles_block += f"    - {r['title']}\n"
        if len(dom_rows) > 0:
            articles_block += f"  국내({len(dom_rows)}건):\n"
            for _, r in dom_rows.iterrows():
                articles_block += f"    - {r['title']}\n"

    cat_keys = ', '.join(f'"{c}"' for c in active_cats)

    prompt = f"""분석 대상일: {td_fmt}
HIGH+MEDIUM 기사: 총 {len(hm)}건 (해외 {len(hm_intl)}건 / 국내 {len(hm_dom)}건)

{articles_block}

아래 JSON 형식으로 분석하세요. 활성 카테고리 키: {cat_keys}

{{
  "executive_summary": "오늘의 핵심 동향 3~4문장. 가장 중요한 변화·위험 신호 중심. 구체적 수치·고유명사 활용.",

  "categories": {{
    "<카테고리코드>": {{
      "overseas": "해외 상황 2~4문장. 국제 기사 기반, 지금 무슨 일이 벌어지고 있는지. 없으면 빈 문자열.",
      "korea_impact": "국내 영향 2~4문장. 국내 기사 기반 한국 산업·경제 파급. 국내 기사 없으면 해외 상황의 한국 파급 가능성 서술."
    }}
  }},

  "flow": {{
    "triggers": {{
      "route":     {{ "overseas": "항로·초크포인트 차단·위협 해외 상황 2~4문장. 없으면 빈 문자열.", "korea_impact": "한국 수입 항로 위협 파급 2~4문장." }},
      "source":    {{ "overseas": "공급원 교란(수출금지·제재·감산) 해외 상황 2~4문장. 없으면 빈 문자열.", "korea_impact": "한국 원자재·에너지 공급원 파급 2~4문장." }},
      "logistics": {{ "overseas": "운임·항만·선박 교란 해외 상황 2~4문장. 없으면 빈 문자열.", "korea_impact": "한국 해운·수출입 물류 파급 2~4문장." }}
    }},
    "domestic_impact": {{
      "경제·금융":  "국내 거시경제·금융시장 영향 2~3문장. 관련 기사 없으면 빈 문자열.",
      "정유·화학":  "정유·석유화학 산업 영향 2~3문장. 없으면 빈 문자열.",
      "해운·물류":  "국내 해운사·물류 업계 영향 2~3문장. 없으면 빈 문자열.",
      "식품·농업":  "식품·농업·수산 영향 2~3문장. 없으면 빈 문자열.",
      "기타산업":   "그 외 주목할 산업 영향 1~2문장. 없으면 빈 문자열."
    }}
  }},

  "changes": {{
    "new":       ["어제 없었다가 오늘 새로 등장한 이슈 (없으면 빈 배열)"],
    "escalated": ["어제보다 심화·확대된 이슈 (없으면 빈 배열)"],
    "resolved":  ["어제 있었으나 오늘 소멸·완화된 이슈 (없으면 빈 배열)"]
  }}
}}

분석 원칙:
- 기사 제목 사실 기반, 과장 없이 현황 중심
- 불확실한 내용은 '~로 보임', '~가능성' 명시
- 한국 경제·산업 시사점 반드시 포함
- JSON 외 다른 텍스트 출력 금지"""

    return prompt, active_cats


def _render_html(td_fmt, hm, hm_intl, hm_domestic, llm_result):
    """공급망 흐름 기반 HTML 렌더링"""

    flow    = llm_result.get('flow', {})
    changes = llm_result.get('changes', {})
    exec_s  = llm_result.get('executive_summary', '')


    def section(title, emoji, content_html):
        return f"""
<div class="section">
  <div class="section-title">{emoji} {title}</div>
  {content_html}
</div>"""

    def flow_block(label, data):
        if not data:
            return ''
        ov = data.get('overseas', '').strip()
        ki = data.get('korea_impact', '').strip()
        if not ov and not ki:
            return ''
        html = f'<div class="flow-item"><div class="flow-label">{label}</div>'
        if ov:
            html += f'<div class="flow-sub"><span class="tag tag-intl">🌐 해외</span><p>{ov}</p></div>'
        if ki:
            html += f'<div class="flow-sub"><span class="tag tag-dom">🇰🇷 국내</span><p>{ki}</p></div>'
        html += '</div>'
        return html

    # 트리거 섹션
    triggers   = flow.get('triggers', {})
    trig_html  = ''
    trig_html += flow_block('경로(ROUTE) 이슈',          triggers.get('route', {}))
    trig_html += flow_block('공급원(SOURCE) 이슈',       triggers.get('source', {}))
    trig_html += flow_block('물류(LOGISTICS) 이슈',      triggers.get('logistics', {}))
    if not trig_html:
        trig_html = '<p class="empty">오늘 주요 국외 트리거 없음</p>'

    # 국내 산업 섹션
    dom_impact = flow.get('domestic_impact', {})
    dom_html   = ''
    for sector, text in dom_impact.items():
        if text and text.strip():
            dom_html += f'<div class="dom-item"><div class="dom-label">{sector}</div><p>{text.strip()}</p></div>'
    if not dom_html:
        dom_html = '<p class="empty">오늘 주요 국내 산업 영향 없음</p>'

    # 어제 대비 변화
    new_items  = changes.get('new', [])
    esc_items  = changes.get('escalated', [])
    res_items  = changes.get('resolved', [])
    chg_html   = ''
    if new_items:
        chg_html += '<div class="chg-group"><span class="chg-label new">신규 ↑</span><ul>'
        for x in new_items: chg_html += f'<li>{x}</li>'
        chg_html += '</ul></div>'
    if esc_items:
        chg_html += '<div class="chg-group"><span class="chg-label esc">심화 ▲</span><ul>'
        for x in esc_items: chg_html += f'<li>{x}</li>'
        chg_html += '</ul></div>'
    if res_items:
        chg_html += '<div class="chg-group"><span class="chg-label res">완화 ↓</span><ul>'
        for x in res_items: chg_html += f'<li>{x}</li>'
        chg_html += '</ul></div>'
    if not chg_html:
        chg_html = '<p class="empty">전일 대비 주요 변화 없음</p>'


    html = f"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>글로벌 공급망 AI 일일 브리핑 — {td_fmt}</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Noto Sans KR', 'Apple SD Gothic Neo', sans-serif; background: #f4f6f9; color: #2c3e50; }}
  .header {{ background: #1a252f; color: white; padding: 24px 32px; }}
  .header h1 {{ font-size: 1.4em; font-weight: 700; margin-bottom: 6px; }}
  .header .meta {{ font-size: 0.85em; color: #aab; }}
  .container {{ max-width: 960px; margin: 0 auto; padding: 24px 16px; }}
  .section {{ background: white; border-radius: 8px; padding: 20px 24px; margin-bottom: 16px; box-shadow: 0 1px 4px rgba(0,0,0,0.08); }}
  .section-title {{ font-size: 1.05em; font-weight: 700; color: #1a252f; margin-bottom: 14px; padding-bottom: 8px; border-bottom: 2px solid #eef0f3; }}
  .exec-text {{ line-height: 1.75; font-size: 0.97em; color: #34495e; }}
  .flow-item {{ margin-bottom: 18px; padding-bottom: 14px; border-bottom: 1px solid #f0f0f0; }}
  .flow-item:last-child {{ border-bottom: none; margin-bottom: 0; }}
  .flow-label {{ font-weight: 700; font-size: 0.9em; color: #2980b9; margin-bottom: 8px; letter-spacing: 0.3px; }}
  .flow-sub {{ display: flex; gap: 10px; margin-bottom: 6px; align-items: flex-start; }}
  .flow-sub p {{ font-size: 0.9em; line-height: 1.65; color: #444; flex: 1; }}
  .tag {{ font-size: 0.78em; font-weight: 700; padding: 2px 8px; border-radius: 4px; white-space: nowrap; margin-top: 2px; }}
  .tag-intl {{ background: #ebf5fb; color: #2980b9; }}
  .tag-dom  {{ background: #eafaf1; color: #27ae60; }}
  .dom-item {{ margin-bottom: 14px; padding-bottom: 10px; border-bottom: 1px solid #f0f0f0; }}
  .dom-item:last-child {{ border-bottom: none; margin-bottom: 0; }}
  .dom-label {{ font-weight: 700; font-size: 0.88em; color: #8e44ad; margin-bottom: 4px; }}
  .dom-item p {{ font-size: 0.9em; line-height: 1.65; color: #444; }}
  .chg-group {{ margin-bottom: 10px; }}
  .chg-label {{ display: inline-block; font-size: 0.8em; font-weight: 700; padding: 2px 10px; border-radius: 4px; margin-bottom: 4px; }}
  .chg-label.new {{ background: #fdebd0; color: #e67e22; }}
  .chg-label.esc {{ background: #fadbd8; color: #c0392b; }}
  .chg-label.res {{ background: #d5f5e3; color: #27ae60; }}
  .chg-group ul {{ padding-left: 18px; }}
  .chg-group li {{ font-size: 0.9em; line-height: 1.65; color: #444; margin-bottom: 2px; }}
  .empty {{ color: #999; font-size: 0.88em; font-style: italic; }}
</style>
</head>
<body>
<div class="header">
  <h1>🚢 글로벌 공급망 AI 일일 브리핑 — {td_fmt}</h1>
  <div class="meta">KMI 해양수산 AX 지원단(hmjeon@kmi.re.kr) &nbsp;|&nbsp; 생성: {datetime.now().strftime("%Y-%m-%d %H:%M")}</div>
</div>
<div class="container">

{section("주요기사 요약", "📌", f'<p class="exec-text">{exec_s}</p>')}

{section("공급망 이슈", "🌐", trig_html)}

{section("국내 산업 영향", "🇰🇷", dom_html)}

{section("어제 대비 변화", "📊", chg_html)}



</div>
</body>
</html>"""
    return html


def generate_llm_report(target_date=None):
    """LLM 1회 호출 → docx(10카테고리) + HTML(공급망흐름) 동시 생성"""

    data = _extract_report_data(target_date)
    if data is None:
        return None, None

    td      = data['td']
    td_fmt  = data['td_fmt']
    hm      = data['hm']

    if len(hm) == 0:
        print("❌ HIGH+MEDIUM 기사 없음 — 리포트 생성 불가")
        return None, None

    print(f"\n🤖 LLM 리포트 생성 중 ({td_fmt}, HIGH+MEDIUM {len(hm)}건)...")

    prompt, active_cats = _build_report_prompt(data)

    # ── Claude API 호출 (1회) ──
    client   = anthropic.Anthropic()
    response = client.messages.create(
        model      = LLM_REPORT_MODEL,
        max_tokens = 16384,
        system     = SYSTEM_PROMPT,
        messages   = [{"role": "user", "content": prompt}]
    )

    raw = response.content[0].text.strip()

    # ── JSON 파싱 ──
    try:
        if '```' in raw:
            raw = raw.split('```')[1]
            if raw.startswith('json'): raw = raw[4:]
            raw = raw.split('```')[0].strip()
        llm_result = _json.loads(raw)
    except _json.JSONDecodeError as e:
        print(f"❌ JSON 파싱 실패: {e}")
        print(f"  Raw (첫 300자):\n{raw[:300]}")
        return None, None

    daily_out_dir = os.path.join(MONITOR_DIR, td)
    os.makedirs(daily_out_dir, exist_ok=True)

    # ══════════════════════════════════════
    # 출력 1: 마크다운 (10카테고리)
    # ══════════════════════════════════════
    lines = []
    L = lines.append
    L(f"# 해상 공급망 모니터링 일일 분석 — {td_fmt}")
    L(f"")
    L(f"> 생성 모델: {LLM_REPORT_MODEL}  |  분석 기준: HIGH+MEDIUM {len(hm)}건 "
      f"(해외 {len(data['hm_intl'])}건 / 국내 {len(data['hm_domestic'])}건)")
    L(f"")
    L("---")
    L("")
    L("## 📌 오늘의 핵심")
    L("")
    L(llm_result.get("executive_summary", ""))
    L("")
    L("---")
    L("")
    L("## 카테고리별 분석")
    L("")
    cats_out = llm_result.get("categories", {})
    written  = 0
    for cat in CAT_ORDER:
        if cat not in cats_out: continue
        cat_data = cats_out[cat]
        cat_kr   = CAT_KR.get(cat, cat)
        s        = data['cat_stats'].get(cat, {})
        h, m     = s.get('high', 0), s.get('med', 0)
        if h + m == 0: continue
        L(f"### {cat_kr}  _(HIGH {h} / MEDIUM {m})_")
        L("")
        overseas     = cat_data.get("overseas", "").strip()
        korea_impact = cat_data.get("korea_impact", "").strip()
        if overseas:
            L("**🌐 해외 상황**"); L(""); L(overseas); L("")
        if korea_impact:
            L("**🇰🇷 국내 영향**"); L(""); L(korea_impact); L("")
        written += 1
    L("---"); L("")
    L("## 통계 요약"); L("")
    L("| 구분 | 건수 | 비중 |"); L("|------|-----:|-----:|")
    for label, cnt in [('총 수집', data['total']), ('HIGH', data['n_high']),
                       ('MEDIUM', data['n_med']), ('LOW', data['n_low']), ('NONE', data['n_none'])]:
        L(f"| {label} | {cnt:,} | {cnt/data['total']*100:.1f}% |")
    L("")
    L("| 카테고리 | HIGH | MEDIUM | 합계 |"); L("|----------|-----:|-------:|-----:|")
    for cat in CAT_ORDER:
        s = data['cat_stats'].get(cat, {})
        if s.get('total', 0) > 0:
            L(f"| {CAT_KR[cat]} | {s['high']} | {s['med']} | {s['total']} |")

    md_path = os.path.join(daily_out_dir, f'daily_report_llm_{td}.md')
    with open(md_path, 'w', encoding='utf-8') as _f:
        _f.write('\n'.join(lines))
    print(f"✅ MD 저장: {md_path}")

    # ══════════════════════════════════════
    # 출력 2: DOCX (10카테고리)
    # ══════════════════════════════════════
    docx_path = None
    try:
        from docx import Document
        from docx.shared import Pt, RGBColor, Cm
        from docx.enum.text import WD_ALIGN_PARAGRAPH

        doc = Document()
        for sec in doc.sections:
            sec.top_margin = Cm(2.5); sec.bottom_margin = Cm(2.5)
            sec.left_margin = Cm(3.0); sec.right_margin = Cm(3.0)

        p = doc.add_heading(f'해상 공급망 모니터링 일일 분석 — {td_fmt}', level=1)
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        meta = doc.add_paragraph()
        meta.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = meta.add_run(f'생성 모델: {LLM_REPORT_MODEL}  |  HIGH+MEDIUM {len(hm)}건 (해외 {len(data["hm_intl"])}건 / 국내 {len(data["hm_domestic"])}건)')
        run.font.size = Pt(9); run.font.color.rgb = RGBColor(0x80, 0x80, 0x80)
        doc.add_paragraph()

        doc.add_heading('오늘의 핵심', level=2)
        doc.add_paragraph(llm_result.get('executive_summary', ''))
        doc.add_paragraph()
        doc.add_heading('카테고리별 분석', level=2)
        for cat in CAT_ORDER:
            if cat not in cats_out: continue
            cat_data = cats_out[cat]
            cat_kr   = CAT_KR.get(cat, cat)
            s        = data['cat_stats'].get(cat, {})
            h, m_cnt = s.get('high', 0), s.get('med', 0)
            if h + m_cnt == 0: continue
            doc.add_heading(f'{cat_kr}  (HIGH {h} / MEDIUM {m_cnt})', level=3)
            overseas     = cat_data.get('overseas', '').strip()
            korea_impact = cat_data.get('korea_impact', '').strip()
            if overseas:
                p = doc.add_paragraph(); p.add_run('🌐 해외 상황').bold = True
                doc.add_paragraph(overseas)
            if korea_impact:
                p = doc.add_paragraph(); p.add_run('🇰🇷 국내 영향').bold = True
                doc.add_paragraph(korea_impact)

        doc.add_page_break()
        doc.add_heading('통계 요약', level=2)
        tbl = doc.add_table(rows=6, cols=3); tbl.style = 'Table Grid'
        for j, h_txt in enumerate(['구분','건수','비중']):
            tbl.rows[0].cells[j].text = h_txt
            tbl.rows[0].cells[j].paragraphs[0].runs[0].bold = True
        for i_r, (label, cnt) in enumerate([('총 수집', data['total']), ('HIGH', data['n_high']),
                                             ('MEDIUM', data['n_med']), ('LOW', data['n_low']), ('NONE', data['n_none'])]):
            tbl.rows[i_r+1].cells[0].text = label
            tbl.rows[i_r+1].cells[1].text = str(cnt)
            tbl.rows[i_r+1].cells[2].text = f"{cnt/data['total']*100:.1f}%"

        docx_path = os.path.join(daily_out_dir, f'daily_report_llm_{td}.docx')
        doc.save(docx_path)
        print(f"✅ DOCX 저장: {docx_path}")
    except ImportError:
        print("⚠ python-docx 미설치 → pip install python-docx")
    except Exception as e:
        print(f"❌ DOCX 생성 실패: {e}")

    # ══════════════════════════════════════
    # 출력 3: JSON (뷰어 재생성용 소스)
    # ══════════════════════════════════════
    json_path = None
    try:
        json_data = {
            'date':     td_fmt,
            'date_key': td,
            'n_total':  int(data['total']),
            'n_high':   int(data['n_high']),
            'n_med':    int(data['n_med']),
            'n_low':    int(data['n_low']),
            'n_none':   int(data['n_none']),
            'n_intl':   int(len(data['hm_intl'])),
            'n_dom':    int(len(data['hm_domestic'])),
            'llm_result': llm_result,
        }
        _out_path = os.path.join(daily_out_dir, f'daily_report_llm_{td}.json')
        with open(_out_path, 'w', encoding='utf-8') as _f:
            _json.dump(json_data, _f, ensure_ascii=False, indent=2)
        json_path = _out_path   # 저장 성공한 경우에만 경로 설정
        print(f"✅ JSON 저장: {json_path}")
    except Exception as e:
        print(f"❌ JSON 저장 실패: {e}")

    # ── 비용 출력 ──
    usage    = response.usage
    cost_in  = usage.input_tokens  * 3  / 1_000_000
    cost_out = usage.output_tokens * 15 / 1_000_000
    print(f"   토큰: 입력 {usage.input_tokens:,} / 출력 {usage.output_tokens:,}")
    print(f"   비용: ${cost_in:.4f} + ${cost_out:.4f} = ${cost_in + cost_out:.4f}")

    return md_path, json_path


In [43]:
# ============================================================
# Cell 12: 리포트 재생성 (날짜 지정, 수집 없이 독립 실행)
# 전제: Cell 1 + Cell 10~11 실행 필요
# ※ 리포트 생성 후 Cell 13(뷰어 빌더)을 실행하면 daily_viewer.html 갱신됨
# ============================================================
# 용도: 이미 분류된 기사(daily CSV 또는 누적 CSV)로 언제든 리포트 재생성
# 전제: Cell 1 + Cell 10~11 실행 필요 (Cell 2~9 수집 건너뛰어도 됨)
#
# 기본값: Cell 1의 d_end 사용. 다른 날짜 재생성 시 REPORT_TARGET_DATE에 날짜 지정.
# ============================================================

from datetime import datetime, timedelta
import os, pandas as pd

# ── 날짜 지정 ──
# 기본값: Cell 1에서 입력한 d_end 자동 사용
# 다른 날짜로 재생성하려면 아래 주석 해제 후 날짜 지정
# REPORT_TARGET_DATE = "2026-03-29"
REPORT_TARGET_DATE = None

if REPORT_TARGET_DATE is None:
    _rd = d_end
else:
    from dateutil.parser import parse as _dp
    _rd = _dp(REPORT_TARGET_DATE).date()
_td = _rd.strftime("%Y%m%d")
_td_fmt = str(_rd)

print(f"=== 리포트 단독 생성 ===")
print(f"대상 날짜: {_td_fmt}")

MONITOR_DIR = MONITOR_DIR if "MONITOR_DIR" in dir() else "monitoring"
CUMUL_DIR   = os.path.join(MONITOR_DIR, "cumulative")

# ── daily CSV 경로 ──
_daily_dir   = os.path.join(MONITOR_DIR, _td)
_gdelt_daily = os.path.join(_daily_dir, f"gdelt_mon_classified_daily_{_td}.csv")
_naver_daily = os.path.join(_daily_dir, f"naver_mon_classified_daily_{_td}.csv")

_dfs     = []
_sources = []
_missing = []

for _csv, _label, _src_tag in [
    (_gdelt_daily, "GDELT",  "GDELT"),
    (_naver_daily, "네이버", "네이버"),
]:
    if os.path.exists(_csv):
        _df = pd.read_csv(_csv, encoding="utf-8-sig")
        _df["_source"] = _src_tag
        _dfs.append(_df)
        _sources.append(f"{_label}({len(_df)}건)")
    else:
        _missing.append(f"{_label}: {_csv}")

if _missing:
    print(f"❌ 분류 CSV 없음 — Cell 1~9 (수집·분류) 먼저 실행하세요.")
    for _m in _missing:
        print(f"   {_m}")

if not _dfs:
    print(f"\n❌ {_td_fmt} 분류 데이터 없음. Cell 1~9 (수집·분류)를 먼저 실행하세요.")
elif _missing:
    print(f"\n⚠ 일부 소스 없음 ({', '.join(m.split(':')[0] for m in _missing)}). 있는 데이터로만 리포트 생성합니다.")
    print(f"📂 로드: {', '.join(_sources)}")
    _md_p, _json_p = generate_llm_report(_rd)
    if _md_p or _json_p:
        print(f"\n✅ 리포트 생성 완료: {_td_fmt}")
        if _md_p:   print(f"   MD  : {_md_p}")
        if _json_p: print(f"   JSON: {_json_p}")
else:
    print(f"📂 로드: {', '.join(_sources)}")
    _md_p, _json_p = generate_llm_report(_rd)
    if _md_p or _json_p:
        print(f"\n✅ 리포트 생성 완료: {_td_fmt}")
        if _md_p:   print(f"   MD  : {_md_p}")
        if _json_p: print(f"   JSON: {_json_p}")


=== 리포트 단독 생성 ===
대상 날짜: 2026-03-30
📂 로드: GDELT(3005건), 네이버(8180건)
📂 로드: GDELT(3005), 네이버(8180)

🤖 LLM 리포트 생성 중 (2026-03-30, HIGH+MEDIUM 3593건)...
✅ MD 저장: monitoring/20260330/daily_report_llm_20260330.md
✅ DOCX 저장: monitoring/20260330/daily_report_llm_20260330.docx
✅ JSON 저장: monitoring/20260330/daily_report_llm_20260330.json
   토큰: 입력 5,891 / 출력 7,779
   비용: $0.0177 + $0.1167 = $0.1344

✅ 리포트 생성 완료: 2026-03-30
   MD  : monitoring/20260330/daily_report_llm_20260330.md
   JSON: monitoring/20260330/daily_report_llm_20260330.json


In [14]:
# ============================================================
# Cell 13: daily_brief.html 빌더 (LLM 없음, JSON → HTML)
# ============================================================
import os, json as _json, glob
from datetime import datetime

VIEWER_PATH = os.path.join(MONITOR_DIR, "daily_brief.html")

pattern    = os.path.join(MONITOR_DIR, "????????", "daily_report_llm_????????.json")
json_files = sorted(glob.glob(pattern), reverse=True)

if not json_files:
    print(f"❌ JSON 없음 — Cell 10~12 실행 후 다시 시도하세요.")
else:
    print(f"📂 발견된 JSON: {len(json_files)}개")
    for jf in json_files: print(f"   {jf}")

    days = []
    for jf in json_files:
        try:
            with open(jf, encoding='utf-8') as _f:
                days.append(_json.load(_f))
        except Exception as e:
            print(f"⚠ 로드 실패 ({jf}): {e}")

    if not days:
        print("❌ 유효한 JSON 없음")
    else:
        def _flow_block(label, data):
            if not data: return ''
            ov = (data.get('overseas') or '').strip()
            ki = (data.get('korea_impact') or '').strip()
            if not ov and not ki: return ''
            html  = f'<div class="flow-item"><div class="flow-label">{label}</div>'
            if ov: html += f'<div class="flow-sub"><span class="tag tag-intl">🌐 해외</span><p>{ov}</p></div>'
            if ki: html += f'<div class="flow-sub"><span class="tag tag-dom">🇰🇷 국내</span><p>{ki}</p></div>'
            html += '</div>'
            return html


        _WDAYS = ['월','화','수','목','금','토','일']
        def _fmt_menu_date(date_str):
            try:
                from datetime import datetime as _dt
                _d = _dt.strptime(date_str, '%Y-%m-%d')
                return f"{_d.year}.{_d.month:02d}.{_d.day:02d} ({_WDAYS[_d.weekday()]})"
            except Exception:
                return date_str

        _WEEKDAYS = ['월요일','화요일','수요일','목요일','금요일','토요일','일요일']
        def _fmt_day_date(date_str):
            try:
                from datetime import datetime as _dt, timedelta
                _d = _dt.strptime(date_str, '%Y-%m-%d')
                _d_pub = _d + timedelta(days=1)
                pub_label = f"{_d_pub.year}.{_d_pub.month:02d}.{_d_pub.day:02d} ({_WEEKDAYS[_d_pub.weekday()]})"
                collect_label = f"{_d.year}.{_d.month:02d}.{_d.day:02d}"
                return f'{pub_label}<span style="font-size:12px; color:#666; margin-left:12px;">기사수집일: {collect_label}</span>'
            except Exception:
                return date_str

        alert_dot = {'CRISIS':'🔴','WARNING':'🟠','CAUTION':'🟡','NORMAL':'🟢'}

        def _render_day(d, idx):
            date_str = d.get('date', '?')
            n_total  = d.get('n_total', 0)
            n_high   = d.get('n_high', 0)
            n_med    = d.get('n_med', 0)
            n_intl   = d.get('n_intl', 0)
            n_dom    = d.get('n_dom', 0)
            llm      = d.get('llm_result', {})
            exec_s   = llm.get('executive_summary', '').strip()
            flow     = llm.get('flow', {})
            changes  = llm.get('changes', {})
            cats     = llm.get('categories', {})

            # 공급망 이슈
            triggers  = flow.get('triggers', {})
            trig_html = ''
            trig_html += _flow_block('경로(ROUTE) 이슈',     triggers.get('route', {}))
            trig_html += _flow_block('공급원(SOURCE) 이슈',  triggers.get('source', {}))
            trig_html += _flow_block('물류(LOGISTICS) 이슈', triggers.get('logistics', {}))
            if not trig_html:
                trig_html = '<p class="empty">오늘 주요 국외 트리거 없음</p>'

            # 국내 산업 영향
            dom_impact = flow.get('domestic_impact', {})
            dom_html   = ''
            for sector, text in dom_impact.items():
                t = (text or '').strip()
                if t: dom_html += f'<div class="dom-item"><div class="dom-label">{sector}</div><p>{t}</p></div>'
            if not dom_html:
                dom_html = '<p class="empty">오늘 주요 국내 산업 영향 없음</p>'

            # 어제 대비 변화
            new_items = changes.get('new', []) or []
            esc_items = changes.get('escalated', []) or []
            res_items = changes.get('resolved', []) or []
            chg_html  = ''
            if new_items:
                chg_html += '<div class="chg-group"><span class="chg-label new">신규 ↑</span><ul>'
                for x in new_items: chg_html += f'<li>{x}</li>'
                chg_html += '</ul></div>'
            if esc_items:
                chg_html += '<div class="chg-group"><span class="chg-label esc">심화 ▲</span><ul>'
                for x in esc_items: chg_html += f'<li>{x}</li>'
                chg_html += '</ul></div>'
            if res_items:
                chg_html += '<div class="chg-group"><span class="chg-label res">완화 ↓</span><ul>'
                for x in res_items: chg_html += f'<li>{x}</li>'
                chg_html += '</ul></div>'
            if not chg_html:
                chg_html = '<p class="empty">전일 대비 주요 변화 없음</p>'

            # 카테고리별 분석
            CAT_KR_MAP = {
                "1_Security":"안보·군사","2_Safety":"해양안전","3_Freight":"운임·물류비",
                "4_PortCargo":"항만·화물","5_EconFinance":"경제·금융","6_Seafood":"수산물",
                "7_Shipping":"해운","8_Logistics":"물류","9_PortCongestion":"항만혼잡",
                "10_OtherIndustry":"기타 산업",
            }
            cat_html = ''
            for cat_key, cat_data in cats.items():
                if not cat_data: continue
                ov = (cat_data.get('overseas') or '').strip()
                ki = (cat_data.get('korea_impact') or '').strip()
                if not ov and not ki: continue
                label     = CAT_KR_MAP.get(cat_key, cat_key)
                cat_html += f'<div class="flow-item"><div class="flow-label">{label}</div>'
                if ov: cat_html += f'<div class="flow-sub"><span class="tag tag-intl">🌐 해외</span><p>{ov}</p></div>'
                if ki: cat_html += f'<div class="flow-sub"><span class="tag tag-dom">🇰🇷 국내</span><p>{ki}</p></div>'
                cat_html += '</div>'
            if not cat_html:
                cat_html = '<p class="empty">카테고리 분석 없음</p>'

            generated_at = datetime.now().strftime("%Y-%m-%d %H:%M")

            return f"""
<div class="day-block" id="day_{idx}">
  <div class="day-date-header">🗓️ {_fmt_day_date(date_str)}</div>
  <div class="day-content">
    <div class="section">
      <div class="section-title">📌 주요기사 요약</div>
      <p class="exec-text">{exec_s if exec_s else '<span class="empty">요약 없음</span>'}</p>
    </div>
    <div class="section">
      <div class="section-title">🌐 공급망 이슈</div>
      {trig_html}
    </div>
    <div class="section">
      <div class="section-title">🇰🇷 국내 산업 영향</div>
      {dom_html}
    </div>
    <div class="section">
      <div class="section-title">📊 어제 대비 변화</div>
      {chg_html}
    </div>
    <div class="section">
      <div class="section-title">🗂 카테고리별 분석</div>
      {cat_html}
    </div>
  </div>
</div>"""

        n = len(days)
        tab_css = '\n'.join(
            f'#tab_{i}:checked ~ .sidebar label[for="tab_{i}"] {{ background:#3498db; }}\n'
            f'#tab_{i}:checked ~ .main #day_{i} {{ display:block; }}'
            for i in range(n)
        )
        menu_items = ''
        for i, d in enumerate(days):
            menu_items += (
                f'<label class="menu-item" for="tab_{i}">'
                f'{_fmt_menu_date(d.get("date","?"))}'
                f'</label>\n'
            )
        radio_inputs = '\n'.join(
            f'<input type="radio" name="daytab" id="tab_{i}" hidden{" checked" if i==0 else ""}>'
            for i in range(n)
        )
        day_blocks = '\n'.join(_render_day(d, i) for i, d in enumerate(days))

        html = f"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>글로벌 공급망 AI 일일 브리핑 | KMI</title>
<style>
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
body {{ font-family: 'Noto Sans KR', 'Apple SD Gothic Neo', sans-serif; font-size: 16px; background: #f5f6fa; color: #2c3e50; }}
/* ── 탭 레이아웃 ── */
.container {{ display: flex; min-height: 100vh; }}
.sidebar {{ width: 180px; min-width: 180px; background: #2c3e50; color: #ecf0f1; overflow-y: auto; padding: 10px 0; position: sticky; top: 0; height: 100vh; align-self: flex-start; }}
.sidebar-title {{ padding: 14px 14px 10px; font-size: 12px; font-weight: bold; border-bottom: 1px solid #2c3e50; color: #bdc3c7; line-height: 1.4; }}
.nav-link {{ display:flex; align-items:center; justify-content:space-between; padding:9px 12px; font-size:12px; font-weight:600; color:#ecf0f1; text-decoration:none; border-radius:6px; background:rgba(52,152,219,0.15); border:1px solid rgba(52,152,219,0.3); margin-bottom:8px; }}
.nav-link:hover {{ background:#3498db; border-color:#3498db; color:#fff; }}
.menu-item {{ display: block; padding: 8px 10px; cursor: pointer; font-size: 12px; line-height: 1.5; border-bottom: 1px solid #34495e; color: #ecf0f1; -webkit-user-select: none; user-select: none; transition: .15s; }}
.menu-item:hover {{ background: #34495e; }}
{tab_css}
/* ── 날짜 블록 ── */
.main {{ flex: 1; min-width: 0; overflow-y: visible; padding: 20px 28px; }}
.day-block {{ display: none; }}
/* ── 기존 daily_report_html과 동일한 스타일 ── */
.report-header {{ background: #2c3e50; color: #ecf0f1; padding: 20px 28px; display: flex; align-items: baseline; gap: 12px; flex-wrap: wrap; }}
.report-header .rh-title {{ font-size: 20px; font-weight: 700; white-space: nowrap; }}
.report-header .rh-sub {{ font-size: 13px; color: #bdc3c7; }}
.day-date-header {{ font-size: 22px; font-weight: 700; color: #2c3e50; padding: 14px 0 10px; border-bottom: 2px solid #e0e4ea; margin-bottom: 16px; }}
.day-content {{ }}
.section {{ background: white; border-radius: 8px; padding: 20px 24px; margin-bottom: 16px; box-shadow: 0 1px 4px rgba(0,0,0,0.08); }}
.section-title {{ font-size: 1.05em; font-weight: 700; color: #2c3e50; margin-bottom: 14px; padding-bottom: 8px; border-bottom: 2px solid #eef0f3; }}
.exec-text {{ line-height: 1.75; font-size: 0.97em; color: #34495e; }}
.flow-item {{ margin-bottom: 18px; padding-bottom: 14px; border-bottom: 1px solid #f0f0f0; }}
.flow-item:last-child {{ border-bottom: none; margin-bottom: 0; }}
.flow-label {{ font-weight: 700; font-size: 0.9em; color: #2980b9; margin-bottom: 8px; letter-spacing: 0.3px; }}
.flow-sub {{ display: flex; gap: 10px; margin-bottom: 6px; align-items: flex-start; }}
.flow-sub p {{ font-size: 0.9em; line-height: 1.65; color: #444; flex: 1; }}
.tag {{ font-size: 0.78em; font-weight: 700; padding: 2px 8px; border-radius: 4px; white-space: nowrap; margin-top: 2px; }}
.tag-intl {{ background: #ebf5fb; color: #2980b9; }}
.tag-dom  {{ background: #eafaf1; color: #27ae60; }}
.dom-item {{ margin-bottom: 14px; padding-bottom: 10px; border-bottom: 1px solid #f0f0f0; }}
.dom-item:last-child {{ border-bottom: none; margin-bottom: 0; }}
.dom-label {{ font-weight: 700; font-size: 0.88em; color: #8e44ad; margin-bottom: 4px; }}
.dom-item p {{ font-size: 0.9em; line-height: 1.65; color: #444; }}
.chg-group {{ margin-bottom: 10px; }}
.chg-label {{ display: inline-block; font-size: 0.8em; font-weight: 700; padding: 2px 10px; border-radius: 4px; margin-bottom: 4px; }}
.chg-label.new {{ background: #fdebd0; color: #e67e22; }}
.chg-label.esc {{ background: #fadbd8; color: #c0392b; }}
.chg-label.res {{ background: #d5f5e3; color: #27ae60; }}
.chg-group ul {{ padding-left: 18px; }}
.chg-group li {{ font-size: 0.9em; line-height: 1.65; color: #444; margin-bottom: 2px; }}
.empty {{ color: #999; font-size: 0.88em; font-style: italic; }}
@media (max-width: 768px) {{
  .report-header {{ padding: 8px 12px; }}
  .report-header .rh-title {{ font-size: 16px; }}
  .report-header .rh-sub {{ font-size: 12px; }}
}}
@media (max-width: 768px) {{
  body {{ font-size: 15px; }}
  .container {{ flex-direction: column; }}
  .sidebar {{ width: 100%; min-width: 0; height: auto; position: sticky; top: 0; z-index: 100;
    display: flex; flex-wrap: nowrap; overflow-x: auto; overflow-y: hidden;
    padding: 6px 8px; gap: 4px; -webkit-overflow-scrolling: touch; }}
  .menu-item {{ flex: 0 0 auto; border-radius: 4px; padding: 4px 10px;
    border-bottom: none; border-right: 1px solid #34495e; font-size: 11px; white-space: nowrap; }}
  .nav-link {{ flex: 0 0 auto; border-radius: 4px; padding: 4px 10px;
    border-bottom: none; white-space: nowrap; font-size: 11px; margin-bottom: 0; }}
  .day-content {{ padding: 12px; }}
}}
body{{-webkit-user-select:none;-ms-user-select:none;user-select:none;}}input,textarea{{-webkit-user-select:text;user-select:text;}}
</style>
</head>
<body>
<div class="container">
{radio_inputs}
<div class="sidebar">
  <!-- <a class="nav-link" href="https://hyongmo.github.io/Global-SCM-Monitoring/weekly_report.html">Go To Weekly <span style="font-size:16px;line-height:1">›</span></a> -->
{menu_items}
</div>
<div class="main">
<div class="report-header">
  <span class="rh-title">🚢 글로벌 공급망 AI 일일 브리핑</span>
  <span class="rh-sub">한국해양수산개발원(KMI) 해양수산 AX 지원단 · hmjeon@kmi.re.kr</span>
</div>
<div style="background:#f0f4f8;border-left:4px solid #2980b9;padding:8px 14px;font-size:11px;color:#555;margin:10px 14px 0 14px">
  본 브리핑은 온톨로지 기반 전문가 지식 그래프와 국내외 기사를 기반으로 생성형 AI가 작성한 것으로 KMI의 공식 의견이 아님을 밝힙니다.</div>
{day_blocks}
</div>
</div>
<script>
(function(){{var a=document['querySelector']('.main');if(!a)return;a['style']['touchAction']='pan-y';var b=0x0;a['addEventListener']('touchstart',function(c){{b=c['changedTouches'][0x0]['clientX'];}},{{'passive':!![]}}),a['addEventListener']('touchend',function(c){{var d=b-c['changedTouches'][0x0]['clientX'];if(Math['abs'](d)<0x32)return;var f=document['querySelector']('input[type=radio][hidden]:checked');if(!f)return;var g=Array['from'](document['querySelectorAll']('input[type=radio][hidden][name=\x27'+f['name']+'\x27]')),h=g['indexOf'](f),i=d>0x0?h+0x1:h-0x1;if(i>=0x0&&i<g['length']){{var j=document['querySelector']('label[for=\x22'+g[i]['id']+'\x22]');j&&(j['click'](),window['scrollTo'](0x0,0x0));}}}},{{'passive':!![]}});}}());
</script>
<script>document.addEventListener('contextmenu',function(e){{e.preventDefault();}});document.addEventListener('keydown',function(e){{if((e.ctrlKey||e.metaKey)&&(e.key==='s'||e.key==='u')){{e.preventDefault();}}}});</script>
</body>
</html>"""

        with open(VIEWER_PATH, 'w', encoding='utf-8') as _f:
            _f.write(html)
        print(f"✅ 뷰어 저장: {VIEWER_PATH}")
        print(f"   날짜 수: {n}개  ({days[-1].get('date','?')} ~ {days[0].get('date','?')})")


📂 발견된 JSON: 7개
   monitoring/20260401/daily_report_llm_20260401.json
   monitoring/20260331/daily_report_llm_20260331.json
   monitoring/20260330/daily_report_llm_20260330.json
   monitoring/20260329/daily_report_llm_20260329.json
   monitoring/20260328/daily_report_llm_20260328.json
   monitoring/20260327/daily_report_llm_20260327.json
   monitoring/20260326/daily_report_llm_20260326.json
✅ 뷰어 저장: monitoring/daily_brief.html
   날짜 수: 7개  (2026-03-26 ~ 2026-04-01)
